In [3]:
import os

base = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"

for vp in ["VP001", "VP002"]:  # دو سوژه کافیه
    path = os.path.join(base, vp)
    print(f"\n{vp}:")
    for f in sorted(os.listdir(path)):
        size_mb = os.path.getsize(os.path.join(path, f)) / 1e6
        print(f"  {f}  ({size_mb:.1f} MB)")


VP001:
  VP001_eeg_after_allchannels.png  (0.9 MB)
  VP001_eeg_after_allchannels_zoom10_30.png  (0.9 MB)
  VP001_eeg_before_allchannels.png  (1.0 MB)
  VP001_eeg_before_allchannels_zoom10_30.png  (0.6 MB)
  VP001_eeg_clean_raw.fif  (44.3 MB)
  VP001_eeg_windows.npy  (1693.7 MB)
  VP001_fnirs_before_after.png  (0.1 MB)
  VP001_fnirs_windows.npy  (1067.0 MB)
  VP001_labels.npy  (0.0 MB)
  VP001_metadata.json  (0.0 MB)
  VP001_session_ids.npy  (0.0 MB)
  VP001_trial_ids.npy  (0.0 MB)

VP002:
  VP002_eeg_after_allchannels.png  (0.8 MB)
  VP002_eeg_after_allchannels_zoom10_30.png  (0.8 MB)
  VP002_eeg_before_allchannels.png  (0.8 MB)
  VP002_eeg_before_allchannels_zoom10_30.png  (0.8 MB)
  VP002_eeg_clean_raw.fif  (44.3 MB)
  VP002_eeg_windows.npy  (1714.2 MB)
  VP002_fnirs_before_after.png  (0.2 MB)
  VP002_fnirs_windows.npy  (1079.9 MB)
  VP002_labels.npy  (0.0 MB)
  VP002_metadata.json  (0.0 MB)
  VP002_session_ids.npy  (0.0 MB)
  VP002_trial_ids.npy  (0.0 MB)


In [2]:
import numpy as np, json, os

base = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"
vp   = "VP001"

# شکل fnirs windows
fnirs = np.load(os.path.join(base, vp, f"{vp}_fnirs_windows.npy"), mmap_mode='r')
print("fnirs_windows shape:", fnirs.shape)
print("fnirs_windows dtype:", fnirs.dtype)
print("fnirs_windows min/max:", fnirs.min(), fnirs.max())

# session ids
sess = np.load(os.path.join(base, vp, f"{vp}_session_ids.npy"))
print("\nsession_ids shape:", sess.shape)
print("unique sessions:", np.unique(sess))

# metadata
with open(os.path.join(base, vp, f"{vp}_metadata.json")) as f:
    meta = json.load(f)
print("\nmetadata keys:", list(meta.keys()))
print("metadata:", meta)

fnirs_windows shape: (827, 21, 16, 16, 30, 2)
fnirs_windows dtype: float32
fnirs_windows min/max: -5.932695 15.552639

session_ids shape: (827,)
unique sessions: [1 2 3]

metadata keys: ['subject_id', 'eeg_shape', 'fnirs_shape', 'labels_distribution', 'eeg_window_len', 'eeg_step', 'eeg_epoch_tmax', 'fnirs_num_segments', 'fnirs_window_len', 'fnirs_step', 'sfreq_eeg', 'sfreq_fnirs', 'beer_lambert_applied', 'fnirs_sync_fix_applied', 'normalization', 'trial_ids_saved']
metadata: {'subject_id': 'VP001', 'eeg_shape': [827, 16, 16, 2000], 'fnirs_shape': [827, 21, 16, 16, 30, 2], 'labels_distribution': [269, 279, 279], 'eeg_window_len': 10.0, 'eeg_step': 1.0, 'eeg_epoch_tmax': 40.0, 'fnirs_num_segments': 21, 'fnirs_window_len': 3.0, 'fnirs_step': 1.0, 'sfreq_eeg': 200.0, 'sfreq_fnirs': 10.0, 'beer_lambert_applied': False, 'fnirs_sync_fix_applied': True, 'normalization': 'per-subject z-score (EEG grid + fNIRS grid, before windowing)', 'trial_ids_saved': True}


In [1]:
"""
=============================================================================
  fNIRS Noise & Artifact Detection — adapted for 26-subject N-back dataset
=============================================================================

Dataset structure:
  - 26 subjects (VP001 to VP026), 3 sessions each
  - Modalities: HbO and HbR
  - Sample rate: 10 Hz
  - Task: N-back (0-back, 2-back, 3-back), each trial 40s

Dataset format (from preprocessing):
  {subject_id}_fnirs_windows.npy -> (n_windows, 21, 16, 16, 30, 2)
      dim0: window index
      dim1: 21 segments per window (3s each, 1s step)
      dim2,3: 16x16 spatial grid
      dim4: 30 samples per segment (3s x 10Hz)
      dim5: channel -- 0=HbO, 1=HbR
  {subject_id}_session_ids.npy   -> (n_windows,) session id per window (1,2,3)
  {subject_id}_trial_ids.npy     -> (n_windows,) trial id per window

Signal reconstruction method (fixes an earlier "sawtooth artifact" bug):
  Using ALL heavily-overlapping windows (step=1s, segment=3s) and averaging
  each window into one "sample" created boxcar-averaging artifacts and
  mislabeled the time axis. FIX: use only the FIRST window per trial
  (offset=0), and within it only NON-OVERLAPPING segments (indices
  0,3,6,...,18 -- 7 segments spanning 21s with zero overlap, since
  segment_step=1s and segment_len=3s -> stride 3 = non-overlapping).
  This yields a genuinely continuous 10Hz signal, trial by trial.

Trial-boundary bug fix (this version):
  Because sessions are reconstructed by concatenating independent trials
  (each trial re-baselined during preprocessing), computing drift,
  baseline-shift, and HbO-HbR correlation on the WHOLE concatenated
  session signal incorrectly picks up the natural baseline jump between
  trials as if it were noise. This inflated every session's score by a
  near-constant offset (roughly proportional to trial count) and made
  scores uninformative for ranking subjects. FIX: compute drift,
  baseline-shift, and HbO-HbR correlation PER TRIAL (using
  trial_boundaries) and average/aggregate afterward — exactly mirroring
  the fix already applied to motion-spike detection.

Outputs:
  - PNG + TXT per subject/session
  - Summary CSV/JSON + quality heatmap + group-level distribution plots

Install:
  pip install numpy scipy scikit-learn matplotlib pywavelets statsmodels
=============================================================================
"""

import os
import gc
import json
import csv
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

from scipy import signal, stats
from scipy.signal import welch, find_peaks
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA

try:
    import pywt
    PYWT_AVAILABLE = True
except ImportError:
    PYWT_AVAILABLE = False


# ══════════════════════════════════════════════════════════════════════════
#  0. Configuration
# ══════════════════════════════════════════════════════════════════════════

PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"
OUTPUT_DIR           = r"E:\kntu\Thesis\codes\Results\fnirs_artifact_report"

FS         = 10.0   # Hz -- confirmed from metadata: sfreq_fnirs=10.0
N_SESSIONS = 3

# Detection thresholds — tuned for z-scored, spatially-averaged fNIRS
THRESH_FLAT_STD    = 1e-4
THRESH_KURTOSIS    = 5.0
THRESH_DRIFT       = 0.5    # z-scored data -- threshold relative to std, not mM
THRESH_SPIKE_STD   = 4.0
THRESH_HBR_CORR    = 0.1    # after z-score + spatial mean, weaker corr is normal
THRESH_SNR_DB      = 3.0
CONTAMINATION_ML   = 0.10

ALL_SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]
CH_NAMES     = ["HbO", "HbR"]


# ══════════════════════════════════════════════════════════════════════════
#  1. Data loading — reconstructs a genuinely continuous, non-overlapping
#     10Hz signal per session, and returns trial boundary indices
# ══════════════════════════════════════════════════════════════════════════

def load_subject_fnirs(subject_id: str, session: int):
    """
    Load HbO/HbR for one subject/session as a continuous, non-overlapping
    10Hz signal, reconstructed trial by trial.

    Returns
    -------
    hbo, hbr         : ndarray, shape (1, n_samples)
    trial_boundaries : list[int] -- sample index where each new trial starts
                       (used to exclude natural baseline resets from
                       motion-spike / drift / baseline-shift / correlation
                       computations)
    or (None, None, None) if the file is missing.
    """
    fnirs_path = os.path.join(PROCESSED_DATA_PATH, subject_id,
                              f"{subject_id}_fnirs_windows.npy")
    sess_path  = os.path.join(PROCESSED_DATA_PATH, subject_id,
                              f"{subject_id}_session_ids.npy")
    trial_path = os.path.join(PROCESSED_DATA_PATH, subject_id,
                              f"{subject_id}_trial_ids.npy")

    if not os.path.exists(fnirs_path):
        return None, None, None

    try:
        sess_ids  = np.load(sess_path)  if os.path.exists(sess_path)  else None
        trial_ids = np.load(trial_path) if os.path.exists(trial_path) else None
        if sess_ids is None or trial_ids is None:
            print(f"    [WARN] session_ids or trial_ids missing for {subject_id}")
            return None, None, None

        sess_mask = sess_ids == session
        if sess_mask.sum() == 0:
            return None, None, None

        # keep only the first window of each trial (offset=0), in trial order
        use_mask, seen_trials, trial_order = np.zeros(len(trial_ids), dtype=bool), set(), []
        for idx in np.where(sess_mask)[0]:
            tid = int(trial_ids[idx])
            if tid not in seen_trials:
                seen_trials.add(tid)
                use_mask[idx] = True
                trial_order.append(tid)
        if use_mask.sum() == 0:
            return None, None, None

        fnirs_all     = np.load(fnirs_path, mmap_mode='r')  # (n_win,21,16,16,30,2)
        sel_windows   = np.array(fnirs_all[use_mask])       # (n_trials,21,16,16,30,2)
        sel_trial_ids = trial_ids[use_mask]
        order_map = {tid: i for i, tid in enumerate(sel_trial_ids)}
        sel_windows = sel_windows[[order_map[tid] for tid in trial_order]]

        # non-overlapping segment indices: 0,3,6,...,18 (stride = seg_len/seg_step)
        n_seg_total = sel_windows.shape[1]
        seg_idx = list(range(0, n_seg_total, 3))

        trial_hbo, trial_hbr, trial_boundaries = [], [], []
        cursor = 0
        for tw in sel_windows:                       # tw: (21,16,16,30,2)
            segs = tw[seg_idx]                         # (7,16,16,30,2)
            spatial_mean = segs.mean(axis=(1, 2))       # (7,30,2)
            flat = spatial_mean.reshape(-1, 2)          # (210,2) real continuous 10Hz
            trial_hbo.append(flat[:, 0])
            trial_hbr.append(flat[:, 1])
            cursor += flat.shape[0]
            trial_boundaries.append(cursor)

        hbo = np.concatenate(trial_hbo)[np.newaxis, :]
        hbr = np.concatenate(trial_hbr)[np.newaxis, :]
        trial_boundaries = trial_boundaries[:-1]   # last boundary = end of signal, not needed

        del fnirs_all, sel_windows
        return hbo.astype(np.float32), hbr.astype(np.float32), trial_boundaries

    except Exception as e:
        print(f"    [WARN] Error loading {fnirs_path}: {e}")
        return None, None, None


# ══════════════════════════════════════════════════════════════════════════
#  2. Statistical detection
# ══════════════════════════════════════════════════════════════════════════

class StatisticalDetector:
    """Classic statistical fNIRS artifact-detection methods"""

    def __init__(self, fs: float = FS):
        self.fs = fs

    def check_flat(self, sig: np.ndarray) -> dict:
        std = float(np.std(sig))
        return {"is_flat": std < THRESH_FLAT_STD, "std": std}

    def check_saturation(self, sig: np.ndarray,
                         percentile: float = 99.5,
                         ratio_thresh: float = 0.05) -> dict:
        max_val   = np.percentile(np.abs(sig), percentile)
        sat_ratio = float(np.mean(np.abs(sig) >= 0.95 * max_val))
        return {"is_saturated": sat_ratio > ratio_thresh, "saturation_ratio": sat_ratio}

    def check_snr(self, sig: np.ndarray, noise_band: tuple = (2.0, 4.0)) -> dict:
        nyq = self.fs / 2
        noise_hi = min(noise_band[1], nyq * 0.99)
        if noise_band[0] >= nyq:
            return {"snr_db": 99.0, "low_snr": False}
        f, pxx = welch(sig, fs=self.fs, nperseg=min(256, len(sig) // 2))
        sig_power   = float(np.sum(pxx[f < noise_band[0]]))
        noise_mask  = (f >= noise_band[0]) & (f <= noise_hi)
        noise_power = float(np.sum(pxx[noise_mask])) if noise_mask.any() else 1e-10
        snr_db = 10 * np.log10(sig_power / (noise_power + 1e-10))
        return {"snr_db": float(snr_db), "low_snr": snr_db < THRESH_SNR_DB}

    def check_kurtosis(self, sig: np.ndarray) -> dict:
        kurt = float(kurtosis(sig))
        return {"kurtosis": kurt, "has_spikes": abs(kurt) > THRESH_KURTOSIS}

    def check_zscore(self, sig: np.ndarray, z_thresh: float = 3.5) -> dict:
        z = np.abs(stats.zscore(sig))
        outlier_ratio = float(np.mean(z > z_thresh))
        return {
            "outlier_ratio": outlier_ratio,
            "has_outliers": outlier_ratio > 0.01,
            "n_outliers": int(np.sum(z > z_thresh)),
        }

    def check_drift_per_trial(self, sig: np.ndarray,
                              trial_boundaries: list = None) -> dict:
        """
        FIX: drift must be computed PER TRIAL, not on the whole concatenated
        session signal. Trials are independently baseline-corrected during
        preprocessing, so consecutive trials can have very different
        baselines purely by design — fitting one linear trend across the
        whole session would report this natural baseline jump as "drift",
        inflating the score for every subject roughly proportionally to
        trial count.
        """
        bounds = [0] + list(trial_boundaries or []) + [len(sig)]
        trial_drifts = []
        for i in range(len(bounds) - 1):
            seg = sig[bounds[i]:bounds[i + 1]]
            if len(seg) < 10:
                continue
            t = np.arange(len(seg))
            coef  = np.polyfit(t, seg, 1)
            trend = np.polyval(coef, t)
            trial_drifts.append(float(abs(trend[-1] - trend[0])))
        drift = float(np.mean(trial_drifts)) if trial_drifts else 0.0
        return {
            "drift_magnitude": drift,
            "drift_per_trial": trial_drifts,
            "has_drift": drift > THRESH_DRIFT,
        }

    def check_hjorth(self, sig: np.ndarray) -> dict:
        d1 = np.diff(sig); d2 = np.diff(d1)
        act = float(np.var(sig))
        mob = float(np.sqrt(np.var(d1) / (act + 1e-10)))
        com = float(np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)) / (mob + 1e-10))
        return {"activity": act, "mobility": mob, "complexity": com,
                "high_complexity": com > 2.5}

    def check_hbo_hbr_corr_per_trial(self, hbo_ch: np.ndarray, hbr_ch: np.ndarray,
                                     trial_boundaries: list = None) -> dict:
        """
        FIX: correlation must be computed PER TRIAL, then averaged — not on
        the whole concatenated session. The discontinuity at each trial
        boundary (independent baselines) drags whole-session correlation
        toward zero regardless of the true within-trial HbO-HbR relationship,
        which should be negative for a physiologically normal hemodynamic
        response.
        """
        bounds = [0] + list(trial_boundaries or []) + [len(hbo_ch)]
        corrs = []
        for i in range(len(bounds) - 1):
            h_o = hbo_ch[bounds[i]:bounds[i + 1]]
            h_r = hbr_ch[bounds[i]:bounds[i + 1]]
            if len(h_o) > 10 and np.std(h_o) > 1e-8 and np.std(h_r) > 1e-8:
                r, _ = stats.pearsonr(h_o, h_r)
                corrs.append(float(r))
        corr = float(np.mean(corrs)) if corrs else 0.0
        return {
            "hbo_hbr_corr": corr,
            "hbo_hbr_corr_per_trial": corrs,
            "suspicious": corr > THRESH_HBR_CORR,
        }

    def check_stationarity(self, sig: np.ndarray) -> dict:
        half = len(sig) // 2
        v1, v2 = float(np.var(sig[:half])), float(np.var(sig[half:]))
        ratio  = max(v1, v2) / (min(v1, v2) + 1e-10)
        return {"variance_ratio": ratio, "is_nonstationary": ratio > 3.0}

    def run_all(self, hbo: np.ndarray, hbr: np.ndarray,
               trial_boundaries: list = None) -> dict:
        results = {}
        for ch in range(hbo.shape[0]):
            results[f"ch_{ch}"] = {
                "flat":         self.check_flat(hbo[ch]),
                "saturation":   self.check_saturation(hbo[ch]),
                "snr":          self.check_snr(hbo[ch]),
                "kurtosis":     self.check_kurtosis(hbo[ch]),
                "zscore":       self.check_zscore(hbo[ch]),
                "drift":        self.check_drift_per_trial(hbo[ch], trial_boundaries),
                "hjorth":       self.check_hjorth(hbo[ch]),
                "hbo_hbr_corr": self.check_hbo_hbr_corr_per_trial(hbo[ch], hbr[ch], trial_boundaries),
                "stationarity": self.check_stationarity(hbo[ch]),
            }
        return results


# ══════════════════════════════════════════════════════════════════════════
#  3. Signal-processing based detection
# ══════════════════════════════════════════════════════════════════════════

class SignalProcessingDetector:
    """Artifact detection via signal-processing methods"""

    def __init__(self, fs: float = FS):
        self.fs = fs

    def physiological_band_power(self, sig: np.ndarray) -> dict:
        """
        Physiological band decomposition for fs=10Hz:
          HRF: 0.01-0.07Hz | Mayer waves: 0.07-0.15Hz |
          Respiration: 0.15-0.40Hz | Heartbeat: 0.40-fs/2 Hz
        """
        nyq = self.fs / 2
        f, pxx = welch(sig, fs=self.fs, nperseg=min(256, len(sig) // 2))
        total = float(np.trapz(pxx, f)) + 1e-10
        bands = {
            "hrf":         (0.010, min(0.070, nyq * 0.99)),
            "mayer_waves": (0.070, min(0.150, nyq * 0.99)),
            "respiration": (0.150, min(0.400, nyq * 0.99)),
            "heartbeat":   (0.400, nyq * 0.99),
        }
        powers = {}
        for name, (lo, hi) in bands.items():
            if lo >= nyq:
                continue
            mask = (f >= lo) & (f <= hi)
            powers[name] = float(np.trapz(pxx[mask], f[mask]) / total) if mask.any() else 0.0
        dominant = max(powers, key=powers.get) if powers else "unknown"
        return {
            "band_powers": powers, "dominant_band": dominant,
            "excess_heartbeat":   powers.get("heartbeat", 0)   > 0.30,
            "excess_respiration": powers.get("respiration", 0) > 0.25,
            "excess_mayer":       powers.get("mayer_waves", 0) > 0.30,
        }

    def detect_motion_spikes(self, sig: np.ndarray,
                             exclude_idx: list = None) -> dict:
        """
        Motion-spike detection via derivative thresholding.
        exclude_idx: indices to ignore (trial boundaries — natural baseline
        resets, not real motion artifacts).
        """
        deriv  = np.abs(np.diff(sig))
        thresh = float(np.mean(deriv) + THRESH_SPIKE_STD * np.std(deriv))
        idx, _ = find_peaks(deriv, height=thresh, distance=int(self.fs))

        if exclude_idx:
            excl = np.array(exclude_idx)
            keep = np.array([
                np.min(np.abs(excl - i)) > 1 if len(excl) > 0 else True
                for i in idx
            ]) if len(idx) > 0 else np.array([], dtype=bool)
            idx = idx[keep] if len(idx) > 0 else idx

        rate = len(idx) / (len(sig) / self.fs / 60) if len(sig) > 0 else 0
        return {
            "spike_indices": idx.tolist(),
            "n_spikes": len(idx),
            "has_spikes": len(idx) > 0,
            "spike_rate_per_min": float(rate),
        }

    def detect_baseline_shift_per_trial(self, sig: np.ndarray,
                                        trial_boundaries: list = None,
                                        window_sec: float = 5.0,
                                        min_gap_sec: float = 8.0) -> dict:
        """
        FIX 1 (previous): baseline-shift detection excludes samples near
        trial boundaries (natural, expected baseline resets from per-trial
        preprocessing), not real artifacts.

        FIX 2 (this version): the sliding-window comparison was firing on
        ordinary within-trial HRF fluctuations, not just genuine step
        changes -- with step=window/2 and a low 0.5*std threshold, it was
        essentially detecting the hemodynamic response itself, producing
        10-30 "shifts" per session (2-3 per trial) with only 9 trials.
        Two changes fix this:
          (a) raise the threshold from 0.5*std to 1.0*std -- only report
              genuinely large, step-like changes, not normal HRF ripple.
          (b) apply non-maximum suppression with a minimum gap
              (min_gap_sec) between reported shifts -- keeps only the
              single strongest shift within any min_gap-second window,
              instead of counting every sliding-window position near a
              real shift as a separate event.
        """
        window  = int(window_sec * self.fs)
        min_gap = int(min_gap_sec * self.fs)
        sig_std = float(np.std(sig))
        bounds  = set(trial_boundaries or [])
        step    = max(1, window // 2)

        candidates = []
        for i in range(window, len(sig) - window, step):
            if any(abs(i - b) < window for b in bounds):
                continue   # skip samples near a trial boundary
            m_before = float(np.mean(sig[i - window:i]))
            m_after  = float(np.mean(sig[i:i + window]))
            diff = abs(m_after - m_before)
            if diff > 1.0 * sig_std:   # raised threshold: 0.5 -> 1.0
                candidates.append((i, diff))

        # non-maximum suppression: keep only the strongest shift within
        # any min_gap-second window, discard nearby weaker detections
        candidates.sort(key=lambda x: -x[1])
        kept = []
        for idx, diff in candidates:
            if all(abs(idx - k) >= min_gap for k in kept):
                kept.append(idx)
        shifts = sorted(kept)

        return {
            "shift_indices": shifts,
            "n_shifts": len(shifts),
            "has_baseline_shift": len(shifts) > 0,
        }

    def wavelet_artifact(self, sig: np.ndarray,
                         wavelet: str = "db4", level: int = 4) -> dict:
        if not PYWT_AVAILABLE:
            return {"has_wavelet_artifacts": False, "note": "pywt not installed"}
        try:
            coeffs   = pywt.wavedec(sig, wavelet, level=level)
            energies = [float(np.sum(c ** 2)) for c in coeffs[1:]]
            baseline = float(np.median(energies))
            art_lvls = [i + 1 for i, e in enumerate(energies) if e > 2.5 * baseline]
            return {
                "detail_energies": energies, "artifact_levels": art_lvls,
                "has_wavelet_artifacts": len(art_lvls) > 0,
            }
        except Exception as e:
            return {"has_wavelet_artifacts": False, "error": str(e)}

    def autocorrelation_check(self, sig: np.ndarray, max_lag: int = 30) -> dict:
        x  = sig - sig.mean()
        ac = np.correlate(x, x, mode="full")
        ac = ac[len(ac) // 2:]
        ac = ac[:max_lag] / (ac[0] + 1e-10)
        s10 = float(np.sum(np.abs(ac[1:10])))
        return {"ac_sum_1to10": s10, "is_noisy": s10 < 1.0, "ac_values": ac.tolist()}

    def run_all(self, hbo: np.ndarray, hbr: np.ndarray,
               trial_boundaries: list = None) -> dict:
        results = {}
        for ch in range(hbo.shape[0]):
            results[f"ch_{ch}"] = {
                "band_power":      self.physiological_band_power(hbo[ch]),
                "motion_spikes":   self.detect_motion_spikes(hbo[ch], exclude_idx=trial_boundaries),
                "baseline_shift":  self.detect_baseline_shift_per_trial(hbo[ch], trial_boundaries),
                "wavelet":         self.wavelet_artifact(hbo[ch]),
                "autocorrelation": self.autocorrelation_check(hbo[ch]),
            }
        return results


# ══════════════════════════════════════════════════════════════════════════
#  4. ML-based detection
# ══════════════════════════════════════════════════════════════════════════

def _extract_window_features(seg: np.ndarray, fs: float) -> np.ndarray:
    f, pxx = welch(seg, fs=fs, nperseg=min(64, len(seg) // 2))
    d1  = np.diff(seg)
    nyq = fs / 2

    def band_p(lo, hi):
        m = (f >= lo) & (f <= min(hi, nyq * 0.99))
        return float(np.sum(pxx[m])) if m.any() else 0.0

    return np.array([
        float(np.mean(seg)), float(np.std(seg)), float(kurtosis(seg)), float(skew(seg)),
        float(np.ptp(seg)),
        band_p(0.01, 0.07), band_p(0.07, 0.15), band_p(0.15, 0.40), band_p(0.40, nyq),
        float(np.std(d1)), float(np.mean(np.abs(d1))),
        float(np.var(seg[:len(seg) // 2])), float(np.var(seg[len(seg) // 2:])),
    ], dtype=np.float32)


class MLDetector:
    """Isolation Forest + LOF + Elliptic Envelope"""

    def __init__(self, fs: float = FS, contamination: float = CONTAMINATION_ML,
                 window_sec: float = 10.0, step_sec: float = 5.0):
        self.fs = fs
        self.contamination = contamination
        self.window = int(window_sec * fs)
        self.step   = int(step_sec * fs)

    def _build_features(self, sig: np.ndarray):
        feats, centers = [], []
        for s in range(0, len(sig) - self.window, self.step):
            feats.append(_extract_window_features(sig[s:s + self.window], self.fs))
            centers.append(s + self.window // 2)
        return np.array(feats, dtype=np.float32), np.array(centers)

    def detect(self, sig: np.ndarray) -> dict:
        feats, centers = self._build_features(sig)
        if len(feats) < 3:
            return {"n_anomalies": 0, "anomaly_times_s": [], "centers_s": [],
                    "vote_counts": [], "if_scores": [], "is_noisy_channel": False,
                    "_too_short": True}

        feats = np.nan_to_num(feats, nan=0, posinf=0, neginf=0)
        X_sc  = StandardScaler().fit_transform(feats)

        iso  = IsolationForest(contamination=self.contamination, n_estimators=100, random_state=42)
        p_if = iso.fit_predict(X_sc)

        lof  = LocalOutlierFactor(n_neighbors=min(10, len(X_sc) - 1), contamination=self.contamination)
        p_lof = lof.fit_predict(X_sc)

        try:
            pca_x = PCA(n_components=min(5, X_sc.shape[1], X_sc.shape[0] - 1)).fit_transform(X_sc)
            ee    = EllipticEnvelope(contamination=self.contamination, random_state=42, support_fraction=0.9)
            p_ee  = ee.fit_predict(pca_x)
        except Exception:
            p_ee = np.ones(len(X_sc), dtype=int)

        votes = ((p_if == -1).astype(int) + (p_lof == -1).astype(int) + (p_ee == -1).astype(int))
        majority = votes >= 2

        return {
            "n_anomalies":      int(majority.sum()),
            "anomaly_times_s":  (centers[majority] / self.fs).tolist(),
            "vote_counts":      votes.tolist(),
            "centers_s":        (centers / self.fs).tolist(),
            "if_scores":        iso.score_samples(X_sc).tolist(),
            "is_noisy_channel": float(majority.mean()) > 0.15,
        }

    def run_all(self, hbo: np.ndarray) -> dict:
        return {f"ch_{ch}": self.detect(hbo[ch]) for ch in range(hbo.shape[0])}


# ══════════════════════════════════════════════════════════════════════════
#  5. Overall quality score
# ══════════════════════════════════════════════════════════════════════════

def compute_quality_score(stat_r: dict, sp_r: dict, ml_r: dict, n_ch: int) -> dict:
    """Overall fNIRS quality score (0-100, higher = worse)."""
    score, reasons = 0.0, []

    for ch in range(n_ch):
        k  = f"ch_{ch}"
        cn = CH_NAMES[ch] if ch < len(CH_NAMES) else f"ch{ch}"

        if stat_r[k]["flat"]["is_flat"]:
            score += 30
            reasons.append(f"{cn}: flat channel")
        if stat_r[k]["saturation"]["is_saturated"]:
            score += 20
            reasons.append(f"{cn}: saturation ({stat_r[k]['saturation']['saturation_ratio']:.2%})")
        if stat_r[k]["snr"]["low_snr"]:
            score += 10
            reasons.append(f"{cn}: low SNR ({stat_r[k]['snr']['snr_db']:.1f} dB)")
        if stat_r[k]["kurtosis"]["has_spikes"]:
            score += 8
            reasons.append(f"{cn}: kurtosis={stat_r[k]['kurtosis']['kurtosis']:.1f}")
        if stat_r[k]["drift"]["has_drift"]:
            score += 8
            reasons.append(f"{cn}: drift={stat_r[k]['drift']['drift_magnitude']:.3f} (per-trial)")
        if stat_r[k]["hbo_hbr_corr"]["suspicious"]:
            score += 10
            reasons.append(f"{cn}: HbO-HbR corr={stat_r[k]['hbo_hbr_corr']['hbo_hbr_corr']:.2f} (per-trial)")

        if sp_r[k]["motion_spikes"]["has_spikes"]:
            n_sp = sp_r[k]["motion_spikes"]["n_spikes"]
            score += min(15, n_sp * 2)
            reasons.append(f"{cn}: {n_sp} motion spike(s)")

        if sp_r[k]["baseline_shift"]["has_baseline_shift"]:
            score += 8
            reasons.append(f"{cn}: {sp_r[k]['baseline_shift']['n_shifts']} baseline shift(s) (trial-excluded)")

        bp = sp_r[k]["band_power"]
        if bp["excess_heartbeat"]:
            score += 5
            reasons.append(f"{cn}: excess heartbeat noise")
        if bp["excess_respiration"]:
            score += 3
            reasons.append(f"{cn}: excess respiration noise")

        if k in ml_r and ml_r[k].get("is_noisy_channel", False):
            score += 10
            reasons.append(f"{cn}: ML anomaly")

    label = ("good" if score < 15 else "moderate" if score < 35 else
             "suspicious" if score < 55 else "poor")
    return {"total_score": round(min(score, 100), 1), "quality_label": label, "reasons": reasons}


# ══════════════════════════════════════════════════════════════════════════
#  6. Per-subject visualization
# ══════════════════════════════════════════════════════════════════════════

def visualize_subject(hbo: np.ndarray, hbr: np.ndarray,
                      subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, output_dir: str,
                      trial_boundaries: list = None) -> str:
    """
    Panel 1: HbO+HbR signal (spikes, ML anomalies, trial boundaries)
    Panel 2: PSD
    Panel 3: ML Ensemble Vote + HbO-HbR scatter (always two distinct columns,
             regardless of n_ch — fixes an earlier gridspec-overlap bug)
    """
    n_ch   = hbo.shape[0]
    n_cols = max(n_ch, 2)
    t      = np.arange(hbo.shape[1]) / FS

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(
        f"fNIRS Artifact Report — {subject_id} | Session {session}\n"
        f"Quality: {quality['quality_label']} (score={quality['total_score']}/100)",
        fontsize=13, fontweight="bold", y=0.99
    )
    gs = gridspec.GridSpec(3, n_cols, figure=fig, hspace=0.55, wspace=0.35)

    for ch in range(n_ch):
        k  = f"ch_{ch}"
        cn = CH_NAMES[ch] if ch < len(CH_NAMES) else f"ch{ch}"

        ax1 = fig.add_subplot(gs[0, ch])
        ax1.plot(t, hbo[ch], color="#E05252", lw=0.7, label="HbO", alpha=0.9)
        ax1.plot(t, hbr[ch], color="#4A90D9", lw=0.7, label="HbR", alpha=0.9)

        if trial_boundaries:
            for i, b in enumerate(trial_boundaries):
                if b < len(t):
                    ax1.axvline(t[b], color="gray", ls=":", lw=0.6, alpha=0.6,
                               label="Trial boundary" if i == 0 else None)

        spk_idx = sp_r[k]["motion_spikes"]["spike_indices"]
        if spk_idx:
            spk_idx = np.array(spk_idx)
            valid = spk_idx[spk_idx < len(hbo[ch])]
            if len(valid) > 0:
                ax1.scatter(t[valid], hbo[ch][valid], c="red", s=25, zorder=5, label="Spike")

        if k in ml_r and "anomaly_times_s" in ml_r[k]:
            for at in ml_r[k]["anomaly_times_s"]:
                ax1.axvspan(at, at + 5, alpha=0.1, color="orange")

        ax1.set_title(f"{cn} -- Signal (non-overlapping, real 10Hz)", fontsize=9)
        ax1.set_xlabel("Time (s)"); ax1.set_ylabel("dConc")
        ax1.legend(fontsize=6, loc="upper right")
        ax1.tick_params(labelsize=7)

        ax2 = fig.add_subplot(gs[1, ch])
        f_w, pxx = welch(hbo[ch], fs=FS, nperseg=min(256, len(hbo[ch]) // 2))
        ax2.semilogy(f_w, pxx, color="#E05252", lw=1.0, label="HbO PSD")
        for lo, hi, lbl, clr in [(0.010, 0.070, "HRF", "#AAFFAA"),
                                  (0.070, 0.150, "Mayer", "#FFFFAA"),
                                  (0.150, 0.400, "Resp", "#AAAAFF"),
                                  (0.400, min(FS / 2, 5), "Heart", "#FFAAAA")]:
            if lo < FS / 2:
                ax2.axvspan(lo, min(hi, FS / 2), alpha=0.25, color=clr, label=lbl)
        ax2.set_xlim(0, min(FS / 2, 5))
        ax2.set_title(f"{cn} -- PSD", fontsize=9)
        ax2.set_xlabel("Frequency (Hz)"); ax2.set_ylabel("PSD")
        ax2.legend(fontsize=5, loc="upper right")
        ax2.tick_params(labelsize=7)

    # row 3: always two dedicated columns, independent of n_ch
    ax3 = fig.add_subplot(gs[2, 0])
    ml_ch = ml_r.get("ch_0", {})
    if ml_ch.get("_too_short") or "centers_s" not in ml_ch:
        ax3.text(0.5, 0.5, "ML skipped\n(signal too short)",
                 ha="center", va="center", transform=ax3.transAxes, fontsize=9, color="gray")
        ax3.set_title("ML Ensemble Vote -- N/A", fontsize=9)
    else:
        centers = np.array(ml_ch["centers_s"])
        votes   = np.array(ml_ch["vote_counts"])
        bw = max(1.0, (t[-1] / max(len(centers), 1)) * 0.8) if len(centers) else 1.0
        ax3.bar(centers, votes, width=bw, color=np.where(votes >= 2, "tomato", "steelblue"))
        ax3.axhline(2, color="red", ls="--", label="Majority >= 2/3")
        ax3.set_title("ML Ensemble Vote (HbO ch0)", fontsize=9)
        ax3.set_xlabel("Time (s)"); ax3.set_ylabel("Vote count")
        ax3.legend(fontsize=7)
    ax3.tick_params(labelsize=7)

    ax4 = fig.add_subplot(gs[2, 1])
    ax4.scatter(hbo[0], hbr[0], s=3, alpha=0.35, color="#7B68EE")
    corr = stat_r["ch_0"]["hbo_hbr_corr"]["hbo_hbr_corr"]
    ax4.set_title(f"HbO-HbR Scatter (per-trial avg r={corr:.2f})", fontsize=9)
    ax4.set_xlabel("HbO"); ax4.set_ylabel("HbR")
    ax4.tick_params(labelsize=7)

    path = os.path.join(output_dir, f"{subject_id}_S{session}_fnirs_report.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


# ══════════════════════════════════════════════════════════════════════════
#  7. Text report
# ══════════════════════════════════════════════════════════════════════════

def write_text_report(subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, n_ch: int, output_dir: str) -> str:
    lines = [
        "=" * 60,
        f"fNIRS Artifact Report -- {subject_id} | Session {session}",
        "=" * 60,
        f"Quality: {quality['quality_label']}  (score={quality['total_score']}/100)",
        f"Reasons: {' | '.join(quality['reasons']) or 'no issues'}",
        "",
    ]
    for ch in range(n_ch):
        k  = f"ch_{ch}"
        cn = CH_NAMES[ch] if ch < len(CH_NAMES) else f"ch{ch}"
        s, p = stat_r[k], sp_r[k]
        lines += [
            f"-- {cn} {'-'*45}",
            f"  Flat          : {s['flat']['is_flat']}  (std={s['flat']['std']:.2e})",
            f"  Saturation    : {s['saturation']['is_saturated']}  (ratio={s['saturation']['saturation_ratio']:.3f})",
            f"  SNR           : {s['snr']['snr_db']:.1f} dB  (low={s['snr']['low_snr']})",
            f"  Kurtosis      : {s['kurtosis']['kurtosis']:.2f}  (spikes={s['kurtosis']['has_spikes']})",
            f"  Z-outliers    : {s['zscore']['n_outliers']} points ({s['zscore']['outlier_ratio']:.2%})",
            f"  Drift (per-trial avg): {s['drift']['drift_magnitude']:.3f}",
            f"  HbO-HbR corr (per-trial avg): {s['hbo_hbr_corr']['hbo_hbr_corr']:.2f}  (suspicious={s['hbo_hbr_corr']['suspicious']})",
            f"  Stationarity  : non-stationary={s['stationarity'].get('is_nonstationary', '?')}",
            f"  Motion spikes : {p['motion_spikes']['n_spikes']}  ({p['motion_spikes']['spike_rate_per_min']:.1f}/min)",
            f"  Baseline shift (trial-excluded): {p['baseline_shift']['n_shifts']}",
            f"  Dominant band : {p['band_power']['dominant_band']}",
            f"  Wavelet art.  : {p['wavelet'].get('has_wavelet_artifacts', '?')}",
            f"  ML anomaly    : {ml_r.get(k, {}).get('n_anomalies', '?')} windows",
            "",
        ]
    lines.append("=" * 60)
    path = os.path.join(output_dir, f"{subject_id}_S{session}_fnirs_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


# ══════════════════════════════════════════════════════════════════════════
#  8. Main pipeline — one subject, one session
# ══════════════════════════════════════════════════════════════════════════

def run_subject_session(subject_id: str, session: int, output_dir: str) -> dict | None:
    hbo, hbr, trial_boundaries = load_subject_fnirs(subject_id, session)
    if hbo is None or hbr is None:
        print(f"  [SKIP] {subject_id} S{session} -- file not found")
        return None

    n_ch = hbo.shape[0]
    dur  = hbo.shape[1] / FS
    n_trials = len(trial_boundaries) + 1 if trial_boundaries else 1
    print(f"  {subject_id} S{session}: {n_ch}ch x {hbo.shape[1]} samples "
          f"({dur:.0f}s, {n_trials} trials, non-overlapping) ...", end=" ", flush=True)

    stat_r  = StatisticalDetector(FS).run_all(hbo, hbr, trial_boundaries=trial_boundaries)
    sp_r    = SignalProcessingDetector(FS).run_all(hbo, hbr, trial_boundaries=trial_boundaries)
    ml_r    = MLDetector(FS).run_all(hbo)
    quality = compute_quality_score(stat_r, sp_r, ml_r, n_ch)

    png = visualize_subject(hbo, hbr, subject_id, session, stat_r, sp_r, ml_r,
                            quality, output_dir, trial_boundaries=trial_boundaries)
    txt = write_text_report(subject_id, session, stat_r, sp_r, ml_r, quality, n_ch, output_dir)

    print(f"[{quality['quality_label']}] score={quality['total_score']}")

    del hbo, hbr
    gc.collect()

    return {
        "subject": subject_id, "session": session,
        "quality_score": quality["total_score"], "quality_label": quality["quality_label"],
        "reasons": quality["reasons"], "n_trials": n_trials,
        "hbo_flat":      stat_r["ch_0"]["flat"]["is_flat"],
        "hbo_saturated": stat_r["ch_0"]["saturation"]["is_saturated"],
        "hbo_snr_db":    stat_r["ch_0"]["snr"]["snr_db"],
        "hbo_kurtosis":  stat_r["ch_0"]["kurtosis"]["kurtosis"],
        "hbo_drift":     stat_r["ch_0"]["drift"]["drift_magnitude"],
        "hbo_hbr_corr":  stat_r["ch_0"]["hbo_hbr_corr"]["hbo_hbr_corr"],
        "hbo_n_spikes":  sp_r["ch_0"]["motion_spikes"]["n_spikes"],
        "hbo_n_shifts":  sp_r["ch_0"]["baseline_shift"]["n_shifts"],
        "dominant_band": sp_r["ch_0"]["band_power"]["dominant_band"],
        "ml_n_anomaly_windows": ml_r.get("ch_0", {}).get("n_anomalies", -1),
        "png": png, "txt": txt,
    }


# ══════════════════════════════════════════════════════════════════════════
#  9. Pipeline for all 26 subjects
# ══════════════════════════════════════════════════════════════════════════

def run_all_subjects(subjects: list[str] = ALL_SUBJECTS,
                     sessions: list[int] | None = None,
                     output_dir: str = OUTPUT_DIR) -> list[dict]:
    if sessions is None:
        sessions = list(range(1, N_SESSIONS + 1))
    os.makedirs(output_dir, exist_ok=True)

    print("=" * 60)
    print(f"  fNIRS Artifact Detection -- {len(subjects)} subjects x {len(sessions)} sessions")
    print("=" * 60)

    all_results = []
    for subj in subjects:
        print(f"\n[{subj}]")
        for sess in sessions:
            r = run_subject_session(subj, sess, output_dir)
            if r:
                all_results.append(r)

    if all_results:
        _save_csv(all_results, output_dir)
        _save_json(all_results, output_dir)
        _plot_heatmap(all_results, output_dir)
        _plot_group_summary(all_results, output_dir)

    print(f"\n{'='*60}")
    print(f"  Done -- {len(all_results)} sessions processed -> {output_dir}")
    print("=" * 60)
    return all_results


def _save_csv(results: list[dict], output_dir: str):
    path = os.path.join(output_dir, "fnirs_summary_all.csv")
    skip = {"png", "txt", "reasons"}
    keys = [k for k in results[0] if k not in skip]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in results:
            w.writerow({k: r[k] for k in keys})
    print(f"\n  [CSV] {path}")


def _save_json(results: list[dict], output_dir: str):
    path = os.path.join(output_dir, "fnirs_summary_all.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"  [JSON] {path}")


def _plot_heatmap(results: list[dict], output_dir: str):
    subjects = sorted(set(r["subject"] for r in results))
    sessions = sorted(set(r["session"] for r in results))
    mat = np.full((len(subjects), len(sessions)), np.nan)
    for r in results:
        mat[subjects.index(r["subject"]), sessions.index(r["session"])] = r["quality_score"]

    fig, ax = plt.subplots(figsize=(max(5, len(sessions) * 1.5), max(8, len(subjects) * 0.4)))
    im = ax.imshow(mat, cmap="RdYlGn_r", vmin=0, vmax=70, aspect="auto")
    plt.colorbar(im, ax=ax, label="Artifact Score (higher = worse)")
    ax.set_xticks(range(len(sessions))); ax.set_xticklabels([f"S{s}" for s in sessions])
    ax.set_yticks(range(len(subjects))); ax.set_yticklabels(subjects, fontsize=7)
    for i in range(len(subjects)):
        for j in range(len(sessions)):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i,j]:.0f}", ha="center", va="center", fontsize=7)
    ax.set_title("fNIRS Quality Heatmap\n(green=good | red=poor)", fontsize=11)
    plt.tight_layout()
    path = os.path.join(output_dir, "fnirs_heatmap.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [Heatmap] {path}")


def _plot_group_summary(results: list[dict], output_dir: str):
    snr_vals  = [r["hbo_snr_db"]   for r in results if "hbo_snr_db"  in r]
    kurt_vals = [r["hbo_kurtosis"] for r in results if "hbo_kurtosis" in r]
    spk_vals  = [r["hbo_n_spikes"] for r in results if "hbo_n_spikes" in r]
    corr_vals = [r["hbo_hbr_corr"] for r in results if "hbo_hbr_corr" in r]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("fNIRS Group-Level Quality Summary -- all subjects", fontsize=12)

    axes[0, 0].hist(snr_vals, bins=20, color="steelblue", edgecolor="white")
    axes[0, 0].axvline(THRESH_SNR_DB, color="red", ls="--", label=f"thresh={THRESH_SNR_DB}")
    axes[0, 0].set_title("SNR Distribution (HbO)"); axes[0, 0].set_xlabel("dB")
    axes[0, 0].legend(fontsize=8)

    axes[0, 1].hist(kurt_vals, bins=20, color="darkorange", edgecolor="white")
    axes[0, 1].axvline(THRESH_KURTOSIS, color="red", ls="--", label=f"thresh={THRESH_KURTOSIS}")
    axes[0, 1].set_title("Kurtosis Distribution (HbO)"); axes[0, 1].set_xlabel("Kurtosis")
    axes[0, 1].legend(fontsize=8)

    axes[1, 0].hist(spk_vals, bins=20, color="tomato", edgecolor="white")
    axes[1, 0].set_title("Motion Spike Count Distribution"); axes[1, 0].set_xlabel("# Spikes")

    axes[1, 1].hist(corr_vals, bins=20, color="#7B68EE", edgecolor="white")
    axes[1, 1].axvline(THRESH_HBR_CORR, color="red", ls="--", label=f"thresh={THRESH_HBR_CORR}")
    axes[1, 1].set_title("HbO-HbR Correlation Distribution (per-trial avg)")
    axes[1, 1].set_xlabel("Pearson r")
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(output_dir, "fnirs_group_summary.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [Group Summary] {path}")


# ══════════════════════════════════════════════════════════════════════════
#  Entry point
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Option A: all subjects, all sessions ─────────────────────────────
    results = run_all_subjects()

    # ── Option B: single subject (for testing) ────────────────────────────
    # results = run_all_subjects(subjects=["VP001"], sessions=[1])

    # ── quick terminal summary ───────────────────────────────────────────
    print(f"\n{'Subject':8s} {'S':3s} {'Score':7s} {'Label':10s} {'Reasons'}")
    print("-" * 72)
    for r in sorted(results, key=lambda x: x["quality_score"], reverse=True):
        print(f"{r['subject']:8s} {r['session']:<3d} {r['quality_score']:<7.1f} "
              f"{r['quality_label']:10s} {'; '.join(r['reasons'][:2])}")


  fNIRS Artifact Detection -- 26 subjects x 3 sessions

[VP001]
  VP001 S1: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=16.0
  VP001 S2: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=16.0
  VP001 S3: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=16.0

[VP002]
  VP002 S1: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=17.0
  VP002 S2: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=17.0
  VP002 S3: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [moderate] score=15.0

[VP003]
  VP003 S1: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [good] score=10.0
  VP003 S2: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [good] score=13.0
  VP003 S3: 1ch x 1890 samples (189s, 9 trials, non-overlapping) ... [good] score=8.0

[VP004]
  VP004 S1: 1ch x 1680 samples (168s, 8 trials, non-overlapping) ... [good] score=12.0
  VP004 S2: 1ch 

In [3]:
"""
=============================================================================
  EEG Noise & Artifact Detection — تطبیق‌یافته برای ۲۶ سوژه N-back
=============================================================================

ساختار دیتاست:
  - ۲۶ سوژه (VP001 تا VP026)
  - ۳ session برای هر سوژه
  - فرمت: فایل‌های numpy از preprocessing pipeline موجود
  - نرخ نمونه: 200 Hz (بعد از downsample در preprocessing)
  - کانال‌ها: 28 کانال EEG (بعد از interpolation و ICA)
  - تسک: N-back (0-back، 2-back، 3-back)

خروجی‌ها:
  - گزارش متنی per-subject (txt)
  - نمودار per-subject (PNG)
  - CSV خلاصه همه ۲۶ سوژه

نصب پیش‌نیازها:
  pip install numpy scipy scikit-learn matplotlib pywavelets mne
=============================================================================
"""

import os
import gc
import json
import warnings
import numpy as np
import mne
mne.set_log_level('WARNING')
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

from scipy import signal, stats
from scipy.signal import welch, butter, filtfilt
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import FastICA, PCA
from sklearn.covariance import EllipticEnvelope

# ══════════════════════════════════════════════════════════════════════════════
#  ۰. تنظیمات — اینجا را برای دیتاست خود تنظیم کنید
# ══════════════════════════════════════════════════════════════════════════════

# مسیر اصلی داده‌های پیش‌پردازش‌شده
# فرض: هر سوژه یک پوشه جداگانه دارد مثل VP001/session_1/eeg_preprocessed.npy
# اگر فرمت متفاوت است، تابع load_subject_eeg را ویرایش کنید
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"

# مسیر خروجی گزارش‌ها
OUTPUT_DIR = r"E:\kntu\Thesis\codes\Results\eeg_artifact_report"

# پارامترهای سیگنال — مطابق preprocessing شما
SFREQ      = 200    # Hz — نرخ نمونه بعد از downsample
N_CHANNELS = 28     # تعداد کانال EEG بعد از ICA و interpolation
N_SESSIONS = 3      # تعداد session برای هر سوژه

# نام کانال‌ها — اگر دارید وارد کنید، وگرنه خودکار ساخته می‌شود
# CH_NAMES = ["Fp1", "Fp2", "F7", "F3", ...]  # ← اینجا وارد کنید
CH_NAMES = None  # اگر None باشد خودکار ساخته می‌شود: CH00..CH27

# آستانه‌های تشخیص — مقادیر پیشنهادی برای EEG پس از ICA
THRESH_ZSCORE    = 3.5    # Z-score آستانه آرتیفکت
THRESH_KURTOSIS  = 5.0    # kurtosis آستانه — بالاتر از ۵ مشکوک
THRESH_AMP_UV    = 150    # peak-to-peak μV — آستانه دامنه
THRESH_EMG_RATIO = 2.5    # نسبت توان بالا/پایین باند برای EMG
CONTAMINATION_ML = 0.10   # نسبت anomaly مورد انتظار برای ML models

# آستانه fraction-based برای Z-score/Modified-Z: کانال فقط وقتی bad است که
# بیش از این درصد از کل نمونه‌هایش outlier باشند (نه صرفاً یک نمونه در کل session)
CONTAM_FRAC_THRESH = 0.01   # 1% — معیار استاندارد EEG QC

# لیست سوژه‌ها
ALL_SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]


# ══════════════════════════════════════════════════════════════════════════════
#  ۱. بارگذاری دیتا
#     این تابع را بر اساس ساختار واقعی فایل‌های شما ویرایش کنید
# ══════════════════════════════════════════════════════════════════════════════

def load_subject_eeg(subject_id: str, session: int) -> np.ndarray | None:
    """
    بارگذاری EEG یک سوژه برای یک session خاص.

    فرمت دیتاست:
      {subject_id}_eeg_clean_raw.fif  → سیگنال continuous همه sessions
      {subject_id}_session_ids.npy    → (n_windows,) session هر window (1,2,3)

    روش جداسازی session (از preprocessing کد):
      در preprocessing، session_id = trial_idx // 9 + 1
      یعنی session ها به ترتیب زمانی در فایل fif هستند.
      فایل fif را به سه بخش مساوی تقسیم می‌کنیم:
        session 1 → 0s تا total/3
        session 2 → total/3 تا 2*total/3
        session 3 → 2*total/3 تا آخر
      این تقریب کافی است چون هر session ~600-700s دارد و
      sessions به ترتیب زمانی ضبط شده‌اند.

    Returns
    -------
    data : ndarray شکل (n_channels, n_samples_session) یا None
    """
    import mne
    mne.set_log_level("WARNING")

    fif_path = os.path.join(PROCESSED_DATA_PATH, subject_id,
                            f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(fif_path):
        return None

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose=False)

        global CH_NAMES
        if CH_NAMES is None:
            CH_NAMES = raw.ch_names

        # تقسیم زمانی به سه session مساوی
        total_dur   = raw.times[-1]
        session_dur = total_dur / 3.0
        t_start     = (session - 1) * session_dur
        t_end       = min(session * session_dur, total_dur)

        # crop و load فقط همان بخش زمانی
        raw_crop = raw.copy().crop(tmin=t_start, tmax=t_end)
        raw_crop.load_data()
        data = raw_crop.get_data()   # (n_channels, n_samples_session) — واحد: Volt (پیش‌فرض MNE)

        # ── تبدیل واحد Volt → μV ──────────────────────────────────────────
        # باگ مهم رفع‌شده: MNE همیشه raw.get_data() را در واحد Volt برمی‌گرداند
        # (نه μV)، اما تمام threshold های این اسکریپت (THRESH_FLAT_STD،
        # THRESH_AMP_UV، thresh_uv در detect_eog، و غیره) بر اساس μV نوشته
        # شده‌اند. بدون این تبدیل، سیگنال واقعی EEG در Volt عددی حدود 1e-5
        # دارد که از THRESH_FLAT_STD=1e-4 کوچک‌تر است → هر کانالی به اشتباه
        # "flat" تشخیص داده می‌شود، و PTP همیشه ~0.0 گزارش می‌شود.
        data = data * 1e6   # Volt → μV

        del raw_crop
        return data.astype(np.float32)

    except Exception as e:
        print(f"    [WARN] Error loading {fif_path}: {e}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  ۲. توابع کمکی
# ══════════════════════════════════════════════════════════════════════════════

def get_ch_names(n_ch: int) -> list[str]:
    if CH_NAMES and len(CH_NAMES) == n_ch:
        return CH_NAMES
    return [f"CH{i:02d}" for i in range(n_ch)]


def bandpower(x: np.ndarray, sfreq: int, band: tuple) -> float:
    """توان باند فرکانسی با Welch PSD"""
    f, pxx = welch(x, fs=sfreq, nperseg=min(512, len(x)))
    idx = (f >= band[0]) & (f <= band[1])
    return float(np.trapz(pxx[idx], f[idx]))


def epoch_signal(data: np.ndarray, sfreq: int,
                 epoch_len: float = 2.0, overlap: float = 0.5):
    """تقسیم سیگنال به epoch های همپوشان"""
    step   = int(epoch_len * sfreq * (1 - overlap))
    win    = int(epoch_len * sfreq)
    epochs, starts = [], []
    for s in range(0, data.shape[1] - win + 1, step):
        epochs.append(data[:, s:s + win])
        starts.append(s / sfreq)
    return np.array(epochs), np.array(starts)


# ══════════════════════════════════════════════════════════════════════════════
#  ۳. تشخیص آماری
# ══════════════════════════════════════════════════════════════════════════════

class StatisticalDetector:
    """روش‌های آماری کلاسیک تشخیص آرتیفکت"""

    def __init__(self,
                 z_thresh:    float = THRESH_ZSCORE,
                 kurt_thresh: float = THRESH_KURTOSIS,
                 amp_thresh:  float = THRESH_AMP_UV):
        self.z_thresh    = z_thresh
        self.kurt_thresh = kurt_thresh
        self.amp_thresh  = amp_thresh

    def detect(self, data: np.ndarray, sfreq: int) -> dict:
        n_ch, n_t = data.shape
        r = {}

        # ── Z-Score ──────────────────────────────────────────────────────
        # باگ رفع‌شده: قبلاً "np.any(z > thresh)" یعنی اگه فقط یک نمونه در
        # کل ~10-20 دقیقه سیگنال (حدود 120,000 نمونه) از آستانه رد بشه، کل
        # کانال bad علامت می‌خورد. با این تعداد نمونه، هر EEG واقعی (با چند
        # چشمک یا حرکت جزئی طبیعی) تقریباً حتماً حداقل یک outlier در هر
        # کانال دارد → همه کانال‌ها به اشتباه flag می‌شدند.
        # فیکس: معیار استاندارد EEG QC — درصد نمونه‌های outlier، نه وجود صرف.
        z = np.abs(stats.zscore(data, axis=1))
        frac_bad_z = np.mean(z > self.z_thresh, axis=1)   # per-channel fraction
        r["zscore"] = {
            "contaminated_ch": np.where(frac_bad_z > CONTAM_FRAC_THRESH)[0].tolist(),
            "contaminated_fraction_per_ch": frac_bad_z.tolist(),
            "max_z": float(z.max()),
            "contaminated_samples_frac": float(np.mean(np.any(z > self.z_thresh, axis=0))),
        }

        # ── Modified Z-Score (مقاوم در برابر outlier شدید) ───────────────
        # همان فیکس فوق — fraction-based به‌جای any-sample
        med = np.median(data, axis=1, keepdims=True)
        mad = np.median(np.abs(data - med), axis=1, keepdims=True)
        mz  = 0.6745 * np.abs(data - med) / (mad + 1e-10)
        frac_bad_mz = np.mean(mz > 3.5, axis=1)
        r["modified_zscore"] = {
            "contaminated_ch": np.where(frac_bad_mz > CONTAM_FRAC_THRESH)[0].tolist(),
            "contaminated_fraction_per_ch": frac_bad_mz.tolist(),
            "max_mz": float(mz.max()),
        }

        # ── Kurtosis ──────────────────────────────────────────────────────
        kurt = stats.kurtosis(data, axis=1)
        r["kurtosis"] = {
            "contaminated_ch": np.where(np.abs(kurt) > self.kurt_thresh)[0].tolist(),
            "values": kurt.tolist(),
            "max_abs_kurt": float(np.abs(kurt).max()),
        }

        # ── Peak-to-Peak Amplitude ────────────────────────────────────────
        ptp = np.ptp(data, axis=1)
        r["amplitude"] = {
            "contaminated_ch": np.where(ptp > self.amp_thresh)[0].tolist(),
            "ptp_values_uv": ptp.tolist(),
            "max_ptp_uv": float(ptp.max()),
        }

        # ── Variance Jump (تغییر ناگهانی واریانس) ────────────────────────
        win_vj = int(0.5 * sfreq)
        var_roll = np.array([
            np.var(data[:, max(0, i - win_vj):i + win_vj], axis=1)
            for i in range(0, n_t, win_vj)
        ])
        if var_roll.shape[0] > 1:
            var_z = np.abs(stats.zscore(var_roll, axis=0))
            r["variance_jump"] = {
                "contaminated_windows": np.where(np.any(var_z > 3, axis=1))[0].tolist(),
                "max_var_z": float(var_z.max()),
            }
        else:
            r["variance_jump"] = {"contaminated_windows": [], "max_var_z": 0.0}

        return r


# ══════════════════════════════════════════════════════════════════════════════
#  ۴. تشخیص پردازش سیگنال
# ══════════════════════════════════════════════════════════════════════════════

class SignalProcessingDetector:
    """تشخیص آرتیفکت با روش‌های پردازش سیگنال"""

    def __init__(self, sfreq: int):
        self.sfreq = sfreq

    def detect_powerline(self, data: np.ndarray,
                         freq: float = 50.0, thresh_db: float = 8.0) -> list:
        """نویز برق شهر ۵۰ هرتز"""
        contaminated = []
        for ch, sig in enumerate(data):
            f, pxx = welch(sig, fs=self.sfreq, nperseg=min(1024, len(sig)))
            idx_pl = np.argmin(np.abs(f - freq))
            idx_bg = (f > freq - 5) & (f < freq - 1)
            if not np.any(idx_bg):
                continue
            db_above = 10 * np.log10(
                pxx[idx_pl] / (np.mean(pxx[idx_bg]) + 1e-20))
            if db_above > thresh_db:
                contaminated.append({
                    "ch": ch, "db_above_bg": round(float(db_above), 2)
                })
        return contaminated

    def detect_eog(self, data: np.ndarray, thresh_uv: float = 75.0) -> list:
        """
        آرتیفکت چشم (پلک‌زدن).
        روی کانال‌های frontal (Fp1/Fp2) بهترین عملکرد را دارد.
        اگر CH_NAMES تنظیم شده، کانال‌های Fp را خودکار انتخاب می‌کند.
        """
        ch_names = get_ch_names(data.shape[0])
        fp_indices = [i for i, n in enumerate(ch_names)
                      if any(n.startswith(p) for p in ["Fp", "FP", "fp"])]
        if not fp_indices:
            fp_indices = list(range(min(2, data.shape[0])))

        b, a = butter(4, 4 / (self.sfreq / 2), btype="low")
        blink_events = []
        for ch in fp_indices:
            sig = data[ch]
            filt = filtfilt(b, a, sig)
            peaks, _ = signal.find_peaks(
                np.abs(filt), height=thresh_uv,
                distance=int(0.5 * self.sfreq))
            blink_events.append({
                "ch": ch,
                "ch_name": ch_names[ch],
                "blink_count": len(peaks),
                "blink_times_s": (peaks / self.sfreq).tolist()[:20],
            })
        return blink_events

    def detect_emg(self, data: np.ndarray,
                   thresh_ratio: float = THRESH_EMG_RATIO) -> list:
        """آرتیفکت عضله — نسبت توان بالا/پایین باند"""
        contaminated = []
        for ch, sig in enumerate(data):
            low_p  = bandpower(sig, self.sfreq, (1,  30))
            high_p = bandpower(sig, self.sfreq, (30, min(90, self.sfreq / 2 - 1)))
            ratio  = high_p / (low_p + 1e-10)
            if ratio > thresh_ratio:
                contaminated.append({
                    "ch": ch, "emg_ratio": round(float(ratio), 3)
                })
        return contaminated

    def detect_ecg(self, data: np.ndarray) -> list:
        """آرتیفکت قلب — تشخیص ضربان منظم"""
        ecg_detected = []
        b, a = butter(4,
                      [0.5 / (self.sfreq / 2), 40 / (self.sfreq / 2)],
                      btype="band")
        for ch, sig in enumerate(data):
            filt  = filtfilt(b, a, sig)
            peaks, _ = signal.find_peaks(
                filt, height=np.std(filt) * 4,
                distance=int(0.5 * self.sfreq))
            if len(peaks) > 2:
                ibi = np.diff(peaks) / self.sfreq
                hr  = 60 / (np.mean(ibi) + 1e-10)
                if 40 < hr < 180:
                    ecg_detected.append({
                        "ch": ch, "estimated_hr_bpm": round(float(hr), 1),
                        "n_peaks": len(peaks),
                    })
        return ecg_detected

    def detect_electrode_pop(self, data: np.ndarray,
                             frac_thresh: float = CONTAM_FRAC_THRESH) -> list:
        """
        جهش ناگهانی الکترود.

        باگ رفع‌شده: قبلاً "if len(jumps) > 0" یعنی حتی یک جهش در کل
        ~130,000 نمونه کافی بود که کل کانال flag شود — دقیقاً همان مشکل
        Z-score. با ~650 ثانیه EEG واقعی، وجود حداقل یک جهش دو-مشتقی در
        هر کانال طبیعی است (حرکت جزئی، تماس الکترود لحظه‌ای). فیکس:
        معیار fraction-based — فقط وقتی کانال bad است که درصد معناداری
        از نمونه‌هایش jump داشته باشند.
        """
        pops = []
        for ch, sig in enumerate(data):
            diff2  = np.abs(np.diff(sig, n=2))
            thresh = np.mean(diff2) + 6 * np.std(diff2)
            jumps  = np.where(diff2 > thresh)[0]
            frac   = len(jumps) / len(diff2) if len(diff2) > 0 else 0.0
            if frac > frac_thresh:
                pops.append({
                    "ch": ch, "n_jumps": len(jumps),
                    "jump_fraction": round(float(frac), 5),
                    "jump_samples": jumps.tolist()[:10],
                })
        return pops

    def detect_flat_signal(self, data: np.ndarray,
                           thresh_std: float = 0.05) -> list:
        """سیگنال مسطح — الکترود قطع‌شده"""
        flat_ch = []
        win = int(1.0 * self.sfreq)
        for ch, sig in enumerate(data):
            stds = [np.std(sig[i:i + win])
                    for i in range(0, len(sig) - win, win)]
            if np.mean(stds) < thresh_std:
                flat_ch.append(ch)
        return flat_ch

    def detect_wavelet(self, data: np.ndarray,
                       level: int = 5, thresh_factor: float = 3.0) -> list:
        """تشخیص آرتیفکت موضعی با Wavelet"""
        contaminated = []
        for ch, sig in enumerate(data):
            coeffs = pywt.wavedec(sig, "db4", level=level)
            energies = np.array([np.sum(c ** 2) for c in coeffs[1:]])
            z = np.abs(energies - np.mean(energies)) / (np.std(energies) + 1e-8)
            if np.any(z > thresh_factor):
                contaminated.append({
                    "ch": ch,
                    "suspicious_bands": np.where(z > thresh_factor)[0].tolist(),
                })
        return contaminated

    def apply_ica_check(self, data: np.ndarray) -> dict:
        """
        ICA برای بررسی کیفیت — نه برای حذف آرتیفکت
        (حذف آرتیفکت در preprocessing اصلی انجام شده)
        کمپوننت‌هایی با kurtosis بالا نشان‌دهنده آرتیفکت باقی‌مانده هستند.
        """
        n_comp = min(data.shape[0], 15)
        try:
            ica     = FastICA(n_components=n_comp, random_state=42, max_iter=500)
            sources = ica.fit_transform(data.T)
            kurt_ic = stats.kurtosis(sources, axis=0)
            artifact_comps = np.where(np.abs(kurt_ic) > 5)[0].tolist()
            return {
                "n_artifact_components_remaining": len(artifact_comps),
                "artifact_component_ids": artifact_comps,
                "max_kurtosis": float(np.abs(kurt_ic).max()),
                "status": "ok",
            }
        except Exception as e:
            return {"status": "failed", "error": str(e)}

    def run_all(self, data: np.ndarray) -> dict:
        return {
            "powerline":    self.detect_powerline(data),
            "eog":          self.detect_eog(data),
            "emg":          self.detect_emg(data),
            "ecg":          self.detect_ecg(data),
            "electrode_pop": self.detect_electrode_pop(data),
            "flat":         self.detect_flat_signal(data),
            "wavelet":      self.detect_wavelet(data),
            "ica_residual": self.apply_ica_check(data),
        }


# ══════════════════════════════════════════════════════════════════════════════
#  ۵. تشخیص ML
# ══════════════════════════════════════════════════════════════════════════════

def extract_epoch_features(epoch: np.ndarray, sfreq: int) -> np.ndarray:
    """استخراج ویژگی از یک epoch چندکانالی — (n_channels, n_samples)"""
    feats = []
    for sig in epoch:
        ch_f = [
            float(np.std(sig)),
            float(stats.kurtosis(sig)),
            float(stats.skew(sig)),
            float(np.ptp(sig)),
            float(np.sqrt(np.mean(sig ** 2))),
            bandpower(sig, sfreq, (0.5, 4)),
            bandpower(sig, sfreq, (4, 8)),
            bandpower(sig, sfreq, (8, 13)),
            bandpower(sig, sfreq, (13, 30)),
            bandpower(sig, sfreq, (30, min(90, sfreq / 2 - 1))),
        ]
        feats.extend(ch_f)
    return np.array(feats, dtype=np.float32)


class MLDetector:
    """تشخیص anomaly با یادگیری ماشین"""

    def __init__(self, contamination: float = CONTAMINATION_ML):
        self.contamination = contamination

    def fit_predict(self, data: np.ndarray, sfreq: int) -> dict:
        epochs, times = epoch_signal(data, sfreq, epoch_len=2.0, overlap=0.5)
        X = np.array([extract_epoch_features(ep, sfreq) for ep in epochs],
                     dtype=np.float32)
        X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)

        scaler = StandardScaler()
        X_sc   = scaler.fit_transform(X)

        results = {"_times": times.tolist()}

        # ── Isolation Forest ──────────────────────────────────────────────
        iso      = IsolationForest(contamination=self.contamination,
                                   n_estimators=150, random_state=42)
        iso_pred = iso.fit_predict(X_sc)
        results["isolation_forest"] = {
            "n_anomalies":    int(np.sum(iso_pred == -1)),
            "anomaly_times_s": times[iso_pred == -1].tolist(),
            "scores": iso.score_samples(X_sc).tolist(),
        }

        # ── Local Outlier Factor ──────────────────────────────────────────
        lof      = LocalOutlierFactor(n_neighbors=min(20, len(X_sc) - 1),
                                      contamination=self.contamination)
        lof_pred = lof.fit_predict(X_sc)
        results["lof"] = {
            "n_anomalies":    int(np.sum(lof_pred == -1)),
            "anomaly_times_s": times[lof_pred == -1].tolist(),
        }

        # ── Elliptic Envelope ─────────────────────────────────────────────
        try:
            pca_x = PCA(n_components=min(10, X_sc.shape[1],
                                         X_sc.shape[0] - 1)).fit_transform(X_sc)
            ee      = EllipticEnvelope(contamination=self.contamination,
                                       random_state=42, support_fraction=0.9)
            ee_pred = ee.fit_predict(pca_x)
            results["elliptic_envelope"] = {
                "n_anomalies":    int(np.sum(ee_pred == -1)),
                "anomaly_times_s": times[ee_pred == -1].tolist(),
            }
        except Exception as e:
            results["elliptic_envelope"] = {"n_anomalies": -1, "error": str(e)}

        # ── Ensemble Vote (اکثریت ≥ ۲ از ۳) ─────────────────────────────
        votes = (
            (iso_pred == -1).astype(int) +
            (lof_pred == -1).astype(int) +
            (ee_pred  == -1).astype(int)
            if "n_anomalies" in results["elliptic_envelope"] and
               results["elliptic_envelope"]["n_anomalies"] >= 0
            else (iso_pred == -1).astype(int) + (lof_pred == -1).astype(int)
        )
        results["ensemble"] = {
            "n_anomalies_majority": int(np.sum(votes >= 2)),
            "anomaly_times_s":      times[votes >= 2].tolist(),
            "vote_counts":          votes.tolist(),
        }

        return results


# ══════════════════════════════════════════════════════════════════════════════
#  ۶. امتیازدهی کلی سوژه
# ══════════════════════════════════════════════════════════════════════════════

def compute_subject_quality_score(stat_r: dict, sp_r: dict,
                                  ml_r: dict, n_ch: int) -> dict:
    """
    امتیاز کیفیت EEG یک session (0 تا 100، بالاتر = کیفیت بدتر).
    برای مقایسه سریع بین سوژه‌ها و session ها.
    """
    score = 0.0
    reasons = []

    # آماری
    pct_bad_ch = len(stat_r["zscore"]["contaminated_ch"]) / n_ch
    score += pct_bad_ch * 20
    if pct_bad_ch > 0.2:
        reasons.append(f"{len(stat_r['zscore']['contaminated_ch'])} کانال Z-score بالا")

    if stat_r["kurtosis"]["max_abs_kurt"] > 10:
        score += 15
        reasons.append(f"kurtosis max={stat_r['kurtosis']['max_abs_kurt']:.1f}")

    if stat_r["amplitude"]["max_ptp_uv"] > THRESH_AMP_UV:
        score += 10
        reasons.append(f"دامنه بالا: {stat_r['amplitude']['max_ptp_uv']:.0f} μV")

    # پردازش سیگنال
    if sp_r["powerline"]:
        score += 10
        reasons.append(f"نویز ۵۰Hz در {len(sp_r['powerline'])} کانال")

    if sp_r["emg"]:
        score += 10
        reasons.append(f"EMG در {len(sp_r['emg'])} کانال")

    total_blinks = sum(e["blink_count"] for e in sp_r["eog"])
    if total_blinks > 20:
        score += 5
        reasons.append(f"{total_blinks} پلک")

    if sp_r["flat"]:
        score += 15
        reasons.append(f"{len(sp_r['flat'])} کانال مسطح")

    ica_rem = sp_r["ica_residual"].get("n_artifact_components_remaining", 0)
    if ica_rem > 2:
        score += ica_rem * 3
        reasons.append(f"{ica_rem} کمپوننت ICA آرتیفکت باقی‌مانده")

    # ML
    frac_anomaly = ml_r["ensemble"]["n_anomalies_majority"] / max(1, len(ml_r["_times"]))
    score += frac_anomaly * 20
    if frac_anomaly > 0.15:
        reasons.append(f"{ml_r['ensemble']['n_anomalies_majority']} اپوک anomaly")

    quality_label = (
        "خوب"      if score < 20 else
        "متوسط"    if score < 40 else
        "مشکوک"    if score < 60 else
        "ضعیف"
    )
    return {
        "total_score": round(min(score, 100), 1),
        "quality_label": quality_label,
        "reasons": reasons,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  ۷. ویژوالیزیشن per-subject
# ══════════════════════════════════════════════════════════════════════════════

def visualize_subject(data: np.ndarray, sfreq: int,
                      subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, output_dir: str):
    """
    نمودار جامع برای یک سوژه — یک session.
    ۶ پلات: Raw EEG، PSD، Kurtosis، Isolation Forest، Wavelet، خلاصه.
    """
    ch_names = get_ch_names(data.shape[0])
    t = np.arange(data.shape[1]) / sfreq

    fig = plt.figure(figsize=(18, 14))
    fig.suptitle(
        f"EEG Artifact Report — {subject_id} | Session {session}\n"
        f"کیفیت: {quality['quality_label']} (امتیاز={quality['total_score']}/100)",
        fontsize=13, fontweight="bold", y=0.98
    )
    gs = gridspec.GridSpec(3, 2, figure=fig,
                           hspace=0.45, wspace=0.35)

    # ── ۱. Raw EEG ────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    n_show = min(data.shape[0], 8)   # حداکثر ۸ کانال برای وضوح
    offsets = np.arange(n_show) * float(np.ptp(data[:n_show]) * 0.6 + 1)
    bad_chs = set(stat_r["zscore"]["contaminated_ch"] +
                  stat_r["kurtosis"]["contaminated_ch"])
    for i in range(n_show):
        color = "tomato" if i in bad_chs else "steelblue"
        ax1.plot(t, data[i] + offsets[i], lw=0.4, color=color, alpha=0.8)
    ax1.set_yticks(offsets)
    ax1.set_yticklabels(ch_names[:n_show], fontsize=7)
    ax1.set_xlabel("Time (s)")
    ax1.set_title(f"Raw EEG — {n_show} کانال اول "
                  f"(قرمز = bad channel)", fontsize=10)
    # نشانه‌گذاری anomaly epochs
    for at in ml_r["ensemble"]["anomaly_times_s"]:
        ax1.axvspan(at, at + 2.0, color="red", alpha=0.08)
    ax1.axvspan(0, 0, color="red", alpha=0.2, label="ML Anomaly epoch")
    ax1.legend(loc="upper right", fontsize=7)

    # ── ۲. PSD همه کانال‌ها ──────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    for i, sig in enumerate(data):
        f, pxx = welch(sig, fs=sfreq, nperseg=min(512, len(sig)))
        lw = 1.2 if i in bad_chs else 0.5
        ax2.semilogy(f, pxx, lw=lw,
                     color="tomato" if i in bad_chs else "steelblue",
                     alpha=0.6)
    ax2.axvline(50, color="darkred", ls="--", lw=1.2, label="50 Hz notch")
    ax2.set_xlim(0, min(95, sfreq / 2))
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Power (μV²/Hz)")
    ax2.set_title("PSD — همه کانال‌ها")
    ax2.legend(fontsize=8)

    # ── ۳. Kurtosis per channel ───────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    kurt_vals = np.abs(stat_r["kurtosis"]["values"])
    colors_k  = ["tomato" if k > THRESH_KURTOSIS else "steelblue"
                 for k in kurt_vals]
    ax3.bar(range(len(kurt_vals)), kurt_vals, color=colors_k, edgecolor="none")
    ax3.axhline(THRESH_KURTOSIS, color="red", ls="--",
                label=f"thresh={THRESH_KURTOSIS}")
    ax3.set_xlabel("Channel index")
    ax3.set_ylabel("|Kurtosis|")
    ax3.set_title("Kurtosis per Channel")
    ax3.legend(fontsize=8)

    # ── ۴. Isolation Forest Scores ───────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    times_ml = np.array(ml_r["_times"])
    scores   = np.array(ml_r["isolation_forest"]["scores"])
    thresh_if = np.percentile(scores, 10)
    ax4.plot(times_ml, scores, "o-", ms=2, lw=0.8, color="darkorange")
    ax4.axhline(thresh_if, color="red", ls="--", label="Anomaly threshold")
    ax4.fill_between(times_ml, scores, thresh_if,
                     where=(scores < thresh_if),
                     color="red", alpha=0.25)
    ax4.set_xlabel("Epoch start (s)")
    ax4.set_ylabel("Anomaly Score")
    ax4.set_title("Isolation Forest")
    ax4.legend(fontsize=8)

    # ── ۵. Ensemble Vote ─────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    votes = np.array(ml_r["ensemble"]["vote_counts"])
    ax5.bar(times_ml, votes, width=max(0.5, (t[-1] / len(times_ml)) * 0.8),
            color=np.where(votes >= 2, "tomato", "steelblue"))
    ax5.axhline(2, color="red", ls="--", label="Majority (≥2/3)")
    ax5.set_xlabel("Epoch start (s)")
    ax5.set_ylabel("Vote count")
    ax5.set_title("Ensemble Vote (IF + LOF + EE)")
    ax5.legend(fontsize=8)

    path = os.path.join(output_dir,
                        f"{subject_id}_S{session}_artifact_report.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


# ══════════════════════════════════════════════════════════════════════════════
#  ۸. گزارش متنی per-subject
# ══════════════════════════════════════════════════════════════════════════════

def write_text_report(subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, n_ch: int, output_dir: str):
    ch_names = get_ch_names(n_ch)
    lines    = [
        f"{'='*60}",
        f"EEG Artifact Report — {subject_id} | Session {session}",
        f"{'='*60}",
        f"کیفیت کلی: {quality['quality_label']}  (امتیاز = {quality['total_score']}/100)",
        f"دلایل: {' | '.join(quality['reasons']) if quality['reasons'] else 'بدون مشکل'}",
        "",
        "── آماری ────────────────────────────────────────────────────",
        f"Z-Score بالا: {[ch_names[i] for i in stat_r['zscore']['contaminated_ch']]} (max={stat_r['zscore']['max_z']:.2f})",
        f"Modified Z   : {[ch_names[i] for i in stat_r['modified_zscore']['contaminated_ch']]}",
        f"Kurtosis بالا: {[ch_names[i] for i in stat_r['kurtosis']['contaminated_ch']]} (max={stat_r['kurtosis']['max_abs_kurt']:.2f})",
        f"PTP بالا (>{THRESH_AMP_UV}μV): {[ch_names[i] for i in stat_r['amplitude']['contaminated_ch']]} (max={stat_r['amplitude']['max_ptp_uv']:.1f}μV)",
        f"Variance Jump: {len(stat_r['variance_jump']['contaminated_windows'])} پنجره مشکوک",
        "",
        "── پردازش سیگنال ─────────────────────────────────────────────",
        f"نویز ۵۰Hz: {len(sp_r['powerline'])} کانال آلوده",
        f"EOG/پلک   : {sum(e['blink_count'] for e in sp_r['eog'])} رویداد",
        f"EMG       : {len(sp_r['emg'])} کانال آلوده  (thresh ratio={THRESH_EMG_RATIO})",
        f"ECG       : {len(sp_r['ecg'])} کانال با الگوی قلبی",
        f"Pop/Jump  : {len(sp_r['electrode_pop'])} کانال با جهش",
        f"Flat      : {sp_r['flat']} کانال مسطح",
        f"Wavelet   : {len(sp_r['wavelet'])} کانال مشکوک",
        f"ICA باقی‌مانده: {sp_r['ica_residual'].get('n_artifact_components_remaining', '?')} کمپوننت",
        "",
        "── یادگیری ماشین ─────────────────────────────────────────────",
        f"Isolation Forest: {ml_r['isolation_forest']['n_anomalies']} اپوک anomaly",
        f"LOF             : {ml_r['lof']['n_anomalies']} اپوک anomaly",
        f"Ensemble (≥2/3) : {ml_r['ensemble']['n_anomalies_majority']} اپوک از {len(ml_r['_times'])}",
        "="*60,
    ]
    path = os.path.join(output_dir,
                        f"{subject_id}_S{session}_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


# ══════════════════════════════════════════════════════════════════════════════
#  ۹. پایپ‌لاین اصلی — یک سوژه، یک session
# ══════════════════════════════════════════════════════════════════════════════

def run_subject_session(subject_id: str, session: int,
                        output_dir: str) -> dict | None:
    """
    پایپ‌لاین کامل برای یک سوژه و یک session.
    Returns dict خلاصه یا None اگر دیتا پیدا نشد.
    """
    data = load_subject_eeg(subject_id, session)
    if data is None:
        print(f"  [SKIP] {subject_id} S{session} — فایل پیدا نشد")
        return None

    n_ch = data.shape[0]
    print(f"  {subject_id} S{session}: {n_ch}ch × {data.shape[1]} samples "
          f"({data.shape[1]/SFREQ:.1f}s) ...", end=" ", flush=True)

    # ── اجرای تشخیص ──────────────────────────────────────────────────────
    stat_r = StatisticalDetector().detect(data, SFREQ)
    sp_det = SignalProcessingDetector(SFREQ)
    sp_r   = sp_det.run_all(data)
    ml_r   = MLDetector().fit_predict(data, SFREQ)
    quality = compute_subject_quality_score(stat_r, sp_r, ml_r, n_ch)

    # ── ذخیره گزارش و نمودار ─────────────────────────────────────────────
    png_path = visualize_subject(data, SFREQ,
                                 subject_id, session,
                                 stat_r, sp_r, ml_r, quality, output_dir)
    txt_path = write_text_report(subject_id, session,
                                 stat_r, sp_r, ml_r, quality, n_ch, output_dir)

    print(f"[{quality['quality_label']}] امتیاز={quality['total_score']}")

    # پاکسازی حافظه
    del data
    gc.collect()

    return {
        "subject": subject_id,
        "session": session,
        "quality_score": quality["total_score"],
        "quality_label": quality["quality_label"],
        "reasons": quality["reasons"],
        "n_bad_ch_zscore": len(stat_r["zscore"]["contaminated_ch"]),
        "n_bad_ch_kurtosis": len(stat_r["kurtosis"]["contaminated_ch"]),
        "max_ptp_uv": stat_r["amplitude"]["max_ptp_uv"],
        "n_powerline_ch": len(sp_r["powerline"]),
        "n_emg_ch": len(sp_r["emg"]),
        "n_blinks": sum(e["blink_count"] for e in sp_r["eog"]),
        "n_anomaly_epochs_ensemble": ml_r["ensemble"]["n_anomalies_majority"],
        "ica_residual_components": sp_r["ica_residual"].get("n_artifact_components_remaining", -1),
        "png": png_path,
        "txt": txt_path,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  ۱۰. پایپ‌لاین همه ۲۶ سوژه
# ══════════════════════════════════════════════════════════════════════════════

def run_all_subjects(subjects: list[str] = ALL_SUBJECTS,
                     sessions: list[int] | None = None,
                     output_dir: str = OUTPUT_DIR) -> list[dict]:
    """
    اجرای کامل برای همه سوژه‌ها و session ها.

    Parameters
    ----------
    subjects   : لیست سوژه‌ها (پیشفرض: همه ۲۶)
    sessions   : لیست session ها (پیشفرض: [1, 2, 3])
    output_dir : پوشه خروجی
    """
    if sessions is None:
        sessions = list(range(1, N_SESSIONS + 1))

    os.makedirs(output_dir, exist_ok=True)

    print("=" * 60)
    print(f"  EEG Artifact Detection — {len(subjects)} سوژه × {len(sessions)} session")
    print("=" * 60)

    all_results = []

    for subj in subjects:
        print(f"\n[{subj}]")
        for sess in sessions:
            result = run_subject_session(subj, sess, output_dir)
            if result:
                all_results.append(result)

    # ── CSV خلاصه همه سوژه‌ها ─────────────────────────────────────────────
    if all_results:
        _save_summary_csv(all_results, output_dir)
        _save_summary_json(all_results, output_dir)
        _plot_summary_heatmap(all_results, output_dir)

    print(f"\n{'='*60}")
    print(f"  اتمام — {len(all_results)} session پردازش شد")
    print(f"  خروجی‌ها در: {output_dir}")
    print("=" * 60)

    return all_results


def _save_summary_csv(results: list[dict], output_dir: str):
    """ذخیره CSV خلاصه"""
    import csv
    path = os.path.join(output_dir, "summary_all_subjects.csv")
    if not results:
        return
    fieldnames = [k for k in results[0].keys() if k not in ("png", "txt")]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow({k: r[k] for k in fieldnames})
    print(f"\n  [CSV] {path}")


def _save_summary_json(results: list[dict], output_dir: str):
    """ذخیره JSON کامل"""
    path = os.path.join(output_dir, "summary_all_subjects.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"  [JSON] {path}")


def _plot_summary_heatmap(results: list[dict], output_dir: str):
    """
    Heatmap کیفیت: سوژه × session
    رنگ: سبز = خوب، قرمز = ضعیف
    """
    subjects = sorted(set(r["subject"] for r in results))
    sessions = sorted(set(r["session"] for r in results))

    score_mat = np.full((len(subjects), len(sessions)), np.nan)
    for r in results:
        si = subjects.index(r["subject"])
        sj = sessions.index(r["session"])
        score_mat[si, sj] = r["quality_score"]

    fig, ax = plt.subplots(figsize=(max(6, len(sessions) * 1.5),
                                    max(8, len(subjects) * 0.45)))
    im = ax.imshow(score_mat, cmap="RdYlGn_r", vmin=0, vmax=80, aspect="auto")
    plt.colorbar(im, ax=ax, label="Artifact Score (↑ = بدتر)")

    ax.set_xticks(range(len(sessions)))
    ax.set_xticklabels([f"S{s}" for s in sessions])
    ax.set_yticks(range(len(subjects)))
    ax.set_yticklabels(subjects, fontsize=7)

    for i in range(len(subjects)):
        for j in range(len(sessions)):
            if not np.isnan(score_mat[i, j]):
                ax.text(j, i, f"{score_mat[i,j]:.0f}",
                        ha="center", va="center", fontsize=7,
                        color="black")

    ax.set_title("EEG Quality Heatmap — همه سوژه‌ها\n"
                 "(سبز=خوب | قرمز=ضعیف)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, "summary_heatmap.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [Heatmap] {path}")


# ══════════════════════════════════════════════════════════════════════════════
#  نقطه ورود
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── گزینه A: همه سوژه‌ها و همه session ها ────────────────────────────
    results = run_all_subjects()

    # ── گزینه B: فقط یک سوژه خاص (برای تست) ────────────────────────────
    # results = run_all_subjects(subjects=["VP001"], sessions=[1])

    # ── گزینه C: فقط session های خاص ─────────────────────────────────────
    # results = run_all_subjects(sessions=[1])

    # ── خلاصه سریع در terminal ───────────────────────────────────────────
    print("\nخلاصه کیفیت:")
    print(f"{'Subject':8s} {'Session':8s} {'Score':8s} {'Label':10s} {'Reasons'}")
    print("-" * 70)
    for r in sorted(results, key=lambda x: x["quality_score"], reverse=True):
        print(f"{r['subject']:8s} {r['session']:<8d} {r['quality_score']:<8.1f} "
              f"{r['quality_label']:10s} {'; '.join(r['reasons'][:2])}")


  EEG Artifact Detection — 26 سوژه × 3 session

[VP001]
  VP001 S1: 28ch × 131644 samples (658.2s) ... [متوسط] امتیاز=38.8
  VP001 S2: 28ch × 131644 samples (658.2s) ... [مشکوک] امتیاز=50.8
  VP001 S3: 28ch × 131644 samples (658.2s) ... [خوب] امتیاز=1.9

[VP002]
  VP002 S1: 28ch × 131747 samples (658.7s) ... [خوب] امتیاز=1.9
  VP002 S2: 28ch × 131746 samples (658.7s) ... [خوب] امتیاز=1.9
  VP002 S3: 28ch × 131747 samples (658.7s) ... [خوب] امتیاز=1.6

[VP003]
  VP003 S1: 28ch × 129748 samples (648.7s) ... [خوب] امتیاز=2.0
  VP003 S2: 28ch × 129747 samples (648.7s) ... [متوسط] امتیاز=26.8
  VP003 S3: 28ch × 129748 samples (648.7s) ... [خوب] امتیاز=1.6

[VP004]
  VP004 S1: 28ch × 129657 samples (648.3s) ... [ضعیف] امتیاز=80.5
  VP004 S2: 28ch × 129657 samples (648.3s) ... [ضعیف] امتیاز=81.9
  VP004 S3: 28ch × 129657 samples (648.3s) ... [ضعیف] امتیاز=82.8

[VP005]
  VP005 S1: 28ch × 129702 samples (648.5s) ... [خوب] امتیاز=2.0
  VP005 S2: 28ch × 129703 samples (648.5s) ... [مشکوک] امتیاز

In [4]:
"""
=============================================================================
  EEG Noise & Artifact Detection — adapted for 26-subject N-back dataset
=============================================================================

Dataset structure:
  - 26 subjects (VP001 to VP026)
  - 3 sessions per subject
  - Format: continuous cleaned EEG (.fif) from the preprocessing pipeline
  - Sample rate: 200 Hz (after downsampling in preprocessing)
  - Channels: 28 EEG channels (after interpolation and ICA)
  - Task: N-back (0-back, 2-back, 3-back)

Outputs:
  - Per-subject text report (.txt)
  - Per-subject figure (.png)
  - Summary CSV / JSON across all 26 subjects
  - Quality heatmap (subject x session)
  - NEW: automatic artifact-pattern classification per session, added as
    "artifact_type" / "artifact_type_score" columns in the summary

Install:
  pip install numpy scipy scikit-learn matplotlib pywavelets mne
=============================================================================
"""

import os
import gc
import json
import csv
import warnings
import numpy as np
import mne
mne.set_log_level("WARNING")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

from scipy import signal, stats
from scipy.signal import welch, butter, filtfilt
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import FastICA, PCA
from sklearn.covariance import EllipticEnvelope

# ══════════════════════════════════════════════════════════════════════════
#  0. Configuration — edit these for your dataset
# ══════════════════════════════════════════════════════════════════════════

PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s-onetime ica"
OUTPUT_DIR           = r"E:\kntu\Thesis\codes\Results\eeg_artifact_report"

# Signal parameters — must match your preprocessing
SFREQ      = 200    # Hz — sample rate after downsampling
N_CHANNELS = 28      # EEG channel count after ICA + interpolation
N_SESSIONS = 3

# Channel names — filled in automatically from the .fif file at load time
CH_NAMES = None

# Detection thresholds — reasonable defaults for post-ICA EEG (values in microvolts, µV)
THRESH_ZSCORE    = 3.5    # Z-score artifact threshold
THRESH_KURTOSIS  = 5.0    # kurtosis threshold — above 5 is suspicious
THRESH_AMP_UV    = 150    # peak-to-peak µV — amplitude threshold
THRESH_EMG_RATIO = 2.5    # high/low band power ratio for EMG
CONTAMINATION_ML = 0.10   # expected anomaly fraction for ML models

# Fraction-based threshold for Z-score / Modified-Z / electrode-pop flags:
# a channel is only flagged "bad" if MORE than this fraction of its samples
# are outliers — not merely because a single sample crossed the threshold
# somewhere across the whole (many-minute) recording.
CONTAM_FRAC_THRESH = 0.01   # 1% — standard EEG QC practice

ALL_SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]

# ── Known artifact channel-cluster signatures ───────────────────────────────
# Derived empirically from manual review of VP001, VP005, VP008, VP015 in
# this dataset. Used by classify_artifact_pattern() below.
PATTERN_TEMPORAL_EMG      = {"T7", "T8", "FC5", "FC6", "C3", "C4", "CP5", "CP6"}
PATTERN_FRONTAL_BILATERAL = {"Fp1", "Fp2", "AFF5h", "AFF6h", "F1", "F2", "FC1", "FC2"}


# ══════════════════════════════════════════════════════════════════════════
#  1. Data loading
# ══════════════════════════════════════════════════════════════════════════

def load_subject_eeg(subject_id: str, session: int) -> np.ndarray | None:
    """
    Load EEG for one subject and one session.

    Dataset format:
      {subject_id}_eeg_clean_raw.fif  -> continuous signal, all sessions
      {subject_id}_session_ids.npy    -> (n_windows,) session id per window (1,2,3)

    Session splitting (derived from the preprocessing code):
      In preprocessing, session_id = trial_idx // 9 + 1, i.e. sessions are
      stored in chronological order inside the .fif file. We split the file
      into three equal time segments:
        session 1 -> 0s to total/3
        session 2 -> total/3 to 2*total/3
        session 3 -> 2*total/3 to end
      This is a good approximation since each session lasts ~600-700s and
      sessions were recorded in chronological order.

    Returns
    -------
    data : ndarray, shape (n_channels, n_samples_session), unit = microvolts (µV)
           or None if the file does not exist.
    """
    fif_path = os.path.join(PROCESSED_DATA_PATH, subject_id,
                            f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(fif_path):
        return None

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose=False)

        global CH_NAMES
        if CH_NAMES is None:
            CH_NAMES = raw.ch_names

        # split into three equal time segments
        total_dur   = raw.times[-1]
        session_dur = total_dur / 3.0
        t_start     = (session - 1) * session_dur
        t_end       = min(session * session_dur, total_dur)

        # crop first, THEN load — avoids loading the whole file into RAM
        raw_crop = raw.copy().crop(tmin=t_start, tmax=t_end)
        raw_crop.load_data()
        data = raw_crop.get_data()   # (n_channels, n_samples), unit: Volt (MNE default)

        # ── Volt -> microvolt conversion ────────────────────────────────
        # IMPORTANT FIX: MNE's raw.get_data() always returns data in Volts,
        # but every threshold in this script (THRESH_FLAT_STD equivalent,
        # THRESH_AMP_UV, EOG thresh_uv, etc.) is written assuming µV. Without
        # this conversion, real EEG amplitude (~1e-5 V) is smaller than most
        # thresholds, causing false "flat channel" / "zero amplitude" flags.
        data = data * 1e6   # Volt -> microvolt

        del raw_crop
        return data.astype(np.float32)

    except Exception as e:
        print(f"    [WARN] Error loading {fif_path}: {e}")
        return None


# ══════════════════════════════════════════════════════════════════════════
#  2. Helper functions
# ══════════════════════════════════════════════════════════════════════════

def get_ch_names(n_ch: int) -> list[str]:
    if CH_NAMES and len(CH_NAMES) == n_ch:
        return CH_NAMES
    return [f"CH{i:02d}" for i in range(n_ch)]


def bandpower(x: np.ndarray, sfreq: int, band: tuple) -> float:
    """Band power via Welch PSD"""
    f, pxx = welch(x, fs=sfreq, nperseg=min(512, len(x)))
    idx = (f >= band[0]) & (f <= band[1])
    return float(np.trapz(pxx[idx], f[idx]))


def epoch_signal(data: np.ndarray, sfreq: int,
                 epoch_len: float = 2.0, overlap: float = 0.5):
    """Split a continuous signal into overlapping epochs"""
    step   = int(epoch_len * sfreq * (1 - overlap))
    win    = int(epoch_len * sfreq)
    epochs, starts = [], []
    for s in range(0, data.shape[1] - win + 1, step):
        epochs.append(data[:, s:s + win])
        starts.append(s / sfreq)
    return np.array(epochs), np.array(starts)


# ══════════════════════════════════════════════════════════════════════════
#  3. Statistical detection
# ══════════════════════════════════════════════════════════════════════════

class StatisticalDetector:
    """Classic statistical artifact-detection methods"""

    def __init__(self,
                 z_thresh:    float = THRESH_ZSCORE,
                 kurt_thresh: float = THRESH_KURTOSIS,
                 amp_thresh:  float = THRESH_AMP_UV):
        self.z_thresh    = z_thresh
        self.kurt_thresh = kurt_thresh
        self.amp_thresh  = amp_thresh

    def detect(self, data: np.ndarray, sfreq: int) -> dict:
        n_ch, n_t = data.shape
        r = {}

        # ── Z-Score ──────────────────────────────────────────────────────
        # Fraction-based flagging: a channel is flagged bad only if MORE
        # than CONTAM_FRAC_THRESH of its samples exceed the threshold — not
        # merely because a single sample among ~100k+ crossed it once.
        z = np.abs(stats.zscore(data, axis=1))
        frac_bad_z = np.mean(z > self.z_thresh, axis=1)
        r["zscore"] = {
            "contaminated_ch": np.where(frac_bad_z > CONTAM_FRAC_THRESH)[0].tolist(),
            "contaminated_fraction_per_ch": frac_bad_z.tolist(),
            "max_z": float(z.max()),
            "contaminated_samples_frac": float(np.mean(np.any(z > self.z_thresh, axis=0))),
        }

        # ── Modified Z-Score (robust to extreme outliers) ─────────────────
        med = np.median(data, axis=1, keepdims=True)
        mad = np.median(np.abs(data - med), axis=1, keepdims=True)
        mz  = 0.6745 * np.abs(data - med) / (mad + 1e-10)
        frac_bad_mz = np.mean(mz > 3.5, axis=1)
        r["modified_zscore"] = {
            "contaminated_ch": np.where(frac_bad_mz > CONTAM_FRAC_THRESH)[0].tolist(),
            "contaminated_fraction_per_ch": frac_bad_mz.tolist(),
            "max_mz": float(mz.max()),
        }

        # ── Kurtosis ──────────────────────────────────────────────────────
        kurt = stats.kurtosis(data, axis=1)
        r["kurtosis"] = {
            "contaminated_ch": np.where(np.abs(kurt) > self.kurt_thresh)[0].tolist(),
            "values": kurt.tolist(),
            "max_abs_kurt": float(np.abs(kurt).max()),
        }

        # ── Peak-to-Peak Amplitude ──────────────────────────────────────
        ptp = np.ptp(data, axis=1)
        r["amplitude"] = {
            "contaminated_ch": np.where(ptp > self.amp_thresh)[0].tolist(),
            "ptp_values_uv": ptp.tolist(),
            "max_ptp_uv": float(ptp.max()),
        }

        # ── Variance Jump (sudden variance change) ────────────────────────
        win_vj = int(0.5 * sfreq)
        var_roll = np.array([
            np.var(data[:, max(0, i - win_vj):i + win_vj], axis=1)
            for i in range(0, n_t, win_vj)
        ])
        if var_roll.shape[0] > 1:
            var_z = np.abs(stats.zscore(var_roll, axis=0))
            r["variance_jump"] = {
                "contaminated_windows": np.where(np.any(var_z > 3, axis=1))[0].tolist(),
                "max_var_z": float(var_z.max()),
            }
        else:
            r["variance_jump"] = {"contaminated_windows": [], "max_var_z": 0.0}

        return r


# ══════════════════════════════════════════════════════════════════════════
#  4. Signal-processing based detection
# ══════════════════════════════════════════════════════════════════════════

class SignalProcessingDetector:
    """Artifact detection via classic signal-processing methods"""

    def __init__(self, sfreq: int):
        self.sfreq = sfreq

    def detect_powerline(self, data: np.ndarray,
                         freq: float = 50.0, thresh_db: float = 8.0) -> list:
        """Mains/powerline noise (50 Hz)"""
        contaminated = []
        for ch, sig in enumerate(data):
            f, pxx = welch(sig, fs=self.sfreq, nperseg=min(1024, len(sig)))
            idx_pl = np.argmin(np.abs(f - freq))
            idx_bg = (f > freq - 5) & (f < freq - 1)
            if not np.any(idx_bg):
                continue
            db_above = 10 * np.log10(
                pxx[idx_pl] / (np.mean(pxx[idx_bg]) + 1e-20))
            if db_above > thresh_db:
                contaminated.append({
                    "ch": ch, "db_above_bg": round(float(db_above), 2)
                })
        return contaminated

    def detect_eog(self, data: np.ndarray, thresh_uv: float = 75.0) -> list:
        """
        Eye-movement artifact (blinks).
        Works best on frontal channels (Fp1/Fp2). If CH_NAMES is available,
        Fp channels are auto-selected.
        """
        ch_names = get_ch_names(data.shape[0])
        fp_indices = [i for i, n in enumerate(ch_names)
                      if any(n.startswith(p) for p in ["Fp", "FP", "fp"])]
        if not fp_indices:
            fp_indices = list(range(min(2, data.shape[0])))

        b, a = butter(4, 4 / (self.sfreq / 2), btype="low")
        blink_events = []
        for ch in fp_indices:
            sig = data[ch]
            filt = filtfilt(b, a, sig)
            peaks, _ = signal.find_peaks(
                np.abs(filt), height=thresh_uv,
                distance=int(0.5 * self.sfreq))
            blink_events.append({
                "ch": ch,
                "ch_name": ch_names[ch],
                "blink_count": len(peaks),
                "blink_times_s": (peaks / self.sfreq).tolist()[:20],
            })
        return blink_events

    def detect_emg(self, data: np.ndarray,
                   thresh_ratio: float = THRESH_EMG_RATIO) -> list:
        """Muscle artifact — high/low band power ratio"""
        contaminated = []
        for ch, sig in enumerate(data):
            low_p  = bandpower(sig, self.sfreq, (1,  30))
            high_p = bandpower(sig, self.sfreq, (30, min(90, self.sfreq / 2 - 1)))
            ratio  = high_p / (low_p + 1e-10)
            if ratio > thresh_ratio:
                contaminated.append({
                    "ch": ch, "emg_ratio": round(float(ratio), 3)
                })
        return contaminated

    def detect_ecg(self, data: np.ndarray) -> list:
        """Cardiac artifact — regular heartbeat pattern"""
        ecg_detected = []
        b, a = butter(4,
                      [0.5 / (self.sfreq / 2), 40 / (self.sfreq / 2)],
                      btype="band")
        for ch, sig in enumerate(data):
            filt  = filtfilt(b, a, sig)
            peaks, _ = signal.find_peaks(
                filt, height=np.std(filt) * 4,
                distance=int(0.5 * self.sfreq))
            if len(peaks) > 2:
                ibi = np.diff(peaks) / self.sfreq
                hr  = 60 / (np.mean(ibi) + 1e-10)
                if 40 < hr < 180:
                    ecg_detected.append({
                        "ch": ch, "estimated_hr_bpm": round(float(hr), 1),
                        "n_peaks": len(peaks),
                    })
        return ecg_detected

    def detect_electrode_pop(self, data: np.ndarray,
                             frac_thresh: float = CONTAM_FRAC_THRESH) -> list:
        """
        Sudden electrode "pop"/jump.

        Fraction-based flagging (same fix as Z-score): a channel is only
        flagged bad if more than frac_thresh of its second-derivative
        samples exceed the jump threshold — not merely because a single
        jump occurred somewhere in ~130,000 samples (which is common and
        expected in any real recording).
        """
        pops = []
        for ch, sig in enumerate(data):
            diff2  = np.abs(np.diff(sig, n=2))
            thresh = np.mean(diff2) + 6 * np.std(diff2)
            jumps  = np.where(diff2 > thresh)[0]
            frac   = len(jumps) / len(diff2) if len(diff2) > 0 else 0.0
            if frac > frac_thresh:
                pops.append({
                    "ch": ch, "n_jumps": len(jumps),
                    "jump_fraction": round(float(frac), 5),
                    "jump_samples": jumps.tolist()[:10],
                })
        return pops

    def detect_flat_signal(self, data: np.ndarray,
                           thresh_std: float = 0.05) -> list:
        """Flat signal — disconnected electrode (threshold in µV)"""
        flat_ch = []
        win = int(1.0 * self.sfreq)
        for ch, sig in enumerate(data):
            stds = [np.std(sig[i:i + win])
                    for i in range(0, len(sig) - win, win)]
            if np.mean(stds) < thresh_std:
                flat_ch.append(ch)
        return flat_ch

    def detect_wavelet(self, data: np.ndarray,
                       level: int = 5, thresh_factor: float = 3.0) -> list:
        """Localized artifact detection via Wavelet decomposition"""
        contaminated = []
        for ch, sig in enumerate(data):
            coeffs = pywt.wavedec(sig, "db4", level=level)
            energies = np.array([np.sum(c ** 2) for c in coeffs[1:]])
            z = np.abs(energies - np.mean(energies)) / (np.std(energies) + 1e-8)
            if np.any(z > thresh_factor):
                contaminated.append({
                    "ch": ch,
                    "suspicious_bands": np.where(z > thresh_factor)[0].tolist(),
                })
        return contaminated

    def apply_ica_check(self, data: np.ndarray) -> dict:
        """
        ICA used here purely for quality-checking, NOT for artifact removal
        (removal already happened in the main preprocessing pipeline).
        Components with high kurtosis indicate residual artifacts.
        """
        n_comp = min(data.shape[0], 15)
        try:
            ica     = FastICA(n_components=n_comp, random_state=42, max_iter=500)
            sources = ica.fit_transform(data.T)
            kurt_ic = stats.kurtosis(sources, axis=0)
            artifact_comps = np.where(np.abs(kurt_ic) > 5)[0].tolist()
            return {
                "n_artifact_components_remaining": len(artifact_comps),
                "artifact_component_ids": artifact_comps,
                "max_kurtosis": float(np.abs(kurt_ic).max()),
                "status": "ok",
            }
        except Exception as e:
            return {"status": "failed", "error": str(e)}

    def run_all(self, data: np.ndarray) -> dict:
        return {
            "powerline":     self.detect_powerline(data),
            "eog":           self.detect_eog(data),
            "emg":           self.detect_emg(data),
            "ecg":           self.detect_ecg(data),
            "electrode_pop": self.detect_electrode_pop(data),
            "flat":          self.detect_flat_signal(data),
            "wavelet":       self.detect_wavelet(data),
            "ica_residual":  self.apply_ica_check(data),
        }


# ══════════════════════════════════════════════════════════════════════════
#  5. ML-based detection
# ══════════════════════════════════════════════════════════════════════════

def extract_epoch_features(epoch: np.ndarray, sfreq: int) -> np.ndarray:
    """Extract features from one multi-channel epoch (n_channels, n_samples)"""
    feats = []
    for sig in epoch:
        ch_f = [
            float(np.std(sig)),
            float(stats.kurtosis(sig)),
            float(stats.skew(sig)),
            float(np.ptp(sig)),
            float(np.sqrt(np.mean(sig ** 2))),
            bandpower(sig, sfreq, (0.5, 4)),
            bandpower(sig, sfreq, (4, 8)),
            bandpower(sig, sfreq, (8, 13)),
            bandpower(sig, sfreq, (13, 30)),
            bandpower(sig, sfreq, (30, min(90, sfreq / 2 - 1))),
        ]
        feats.extend(ch_f)
    return np.array(feats, dtype=np.float32)


class MLDetector:
    """Anomaly detection via machine learning"""

    def __init__(self, contamination: float = CONTAMINATION_ML):
        self.contamination = contamination

    def fit_predict(self, data: np.ndarray, sfreq: int) -> dict:
        epochs, times = epoch_signal(data, sfreq, epoch_len=2.0, overlap=0.5)
        X = np.array([extract_epoch_features(ep, sfreq) for ep in epochs],
                     dtype=np.float32)
        X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)

        scaler = StandardScaler()
        X_sc   = scaler.fit_transform(X)

        results = {"_times": times.tolist()}

        # ── Isolation Forest ──────────────────────────────────────────────
        iso      = IsolationForest(contamination=self.contamination,
                                   n_estimators=150, random_state=42)
        iso_pred = iso.fit_predict(X_sc)
        results["isolation_forest"] = {
            "n_anomalies":    int(np.sum(iso_pred == -1)),
            "anomaly_times_s": times[iso_pred == -1].tolist(),
            "scores": iso.score_samples(X_sc).tolist(),
        }

        # ── Local Outlier Factor ──────────────────────────────────────────
        lof      = LocalOutlierFactor(n_neighbors=min(20, len(X_sc) - 1),
                                      contamination=self.contamination)
        lof_pred = lof.fit_predict(X_sc)
        results["lof"] = {
            "n_anomalies":    int(np.sum(lof_pred == -1)),
            "anomaly_times_s": times[lof_pred == -1].tolist(),
        }

        # ── Elliptic Envelope ─────────────────────────────────────────────
        try:
            pca_x = PCA(n_components=min(10, X_sc.shape[1],
                                         X_sc.shape[0] - 1)).fit_transform(X_sc)
            ee      = EllipticEnvelope(contamination=self.contamination,
                                       random_state=42, support_fraction=0.9)
            ee_pred = ee.fit_predict(pca_x)
            results["elliptic_envelope"] = {
                "n_anomalies":    int(np.sum(ee_pred == -1)),
                "anomaly_times_s": times[ee_pred == -1].tolist(),
            }
        except Exception as e:
            results["elliptic_envelope"] = {"n_anomalies": -1, "error": str(e)}

        # ── Ensemble Vote (majority: >= 2 of 3) ────────────────────────────
        votes = (
            (iso_pred == -1).astype(int) +
            (lof_pred == -1).astype(int) +
            (ee_pred  == -1).astype(int)
            if "n_anomalies" in results["elliptic_envelope"] and
               results["elliptic_envelope"]["n_anomalies"] >= 0
            else (iso_pred == -1).astype(int) + (lof_pred == -1).astype(int)
        )
        results["ensemble"] = {
            "n_anomalies_majority": int(np.sum(votes >= 2)),
            "anomaly_times_s":      times[votes >= 2].tolist(),
            "vote_counts":          votes.tolist(),
        }

        return results


# ══════════════════════════════════════════════════════════════════════════
#  6. Overall quality score
# ══════════════════════════════════════════════════════════════════════════

def compute_subject_quality_score(stat_r: dict, sp_r: dict,
                                  ml_r: dict, n_ch: int) -> dict:
    """
    Overall EEG quality score for one session (0-100, higher = worse).
    Used for quick comparison across subjects/sessions.
    """
    score = 0.0
    reasons = []

    # statistical
    pct_bad_ch = len(stat_r["zscore"]["contaminated_ch"]) / n_ch
    score += pct_bad_ch * 20
    if pct_bad_ch > 0.2:
        reasons.append(f"{len(stat_r['zscore']['contaminated_ch'])} channels high Z-score")

    if stat_r["kurtosis"]["max_abs_kurt"] > 10:
        score += 15
        reasons.append(f"kurtosis max={stat_r['kurtosis']['max_abs_kurt']:.1f}")

    if stat_r["amplitude"]["max_ptp_uv"] > THRESH_AMP_UV:
        score += 10
        reasons.append(f"high amplitude: {stat_r['amplitude']['max_ptp_uv']:.0f} uV")

    # signal processing
    if sp_r["powerline"]:
        score += 10
        reasons.append(f"50Hz noise in {len(sp_r['powerline'])} channels")

    if sp_r["emg"]:
        score += 10
        reasons.append(f"EMG in {len(sp_r['emg'])} channels")

    total_blinks = sum(e["blink_count"] for e in sp_r["eog"])
    if total_blinks > 20:
        score += 5
        reasons.append(f"{total_blinks} blinks")

    if sp_r["flat"]:
        score += 15
        reasons.append(f"{len(sp_r['flat'])} flat channels")

    ica_rem = sp_r["ica_residual"].get("n_artifact_components_remaining", 0)
    if ica_rem > 2:
        score += ica_rem * 3
        reasons.append(f"{ica_rem} residual ICA artifact components")

    # ML
    frac_anomaly = ml_r["ensemble"]["n_anomalies_majority"] / max(1, len(ml_r["_times"]))
    score += frac_anomaly * 20
    if frac_anomaly > 0.15:
        reasons.append(f"{ml_r['ensemble']['n_anomalies_majority']} anomaly epochs")

    quality_label = (
        "good"        if score < 20 else
        "moderate"    if score < 40 else
        "suspicious"  if score < 60 else
        "poor"
    )
    return {
        "total_score": round(min(score, 100), 1),
        "quality_label": quality_label,
        "reasons": reasons,
    }


# ══════════════════════════════════════════════════════════════════════════
#  6b. Artifact-pattern classification  (NEW)
# ══════════════════════════════════════════════════════════════════════════

def classify_artifact_pattern(kurt_bad_ch_names: list, zscore_bad_ch_names: list,
                              kurt_values: list, n_ch: int) -> tuple[str, float]:
    """
    Classify the dominant artifact pattern for a session, based on which
    channels were flagged by kurtosis / Z-score.

    Categories (empirically derived from manual review of VP001, VP005,
    VP008, VP015 in this dataset):
      - "temporal_emg"         : flagged channels overlap strongly with
                                  {T7,T8,FC5,FC6,C3,C4,CP5,CP6}
                                  (jaw / temporalis muscle tension)
      - "frontal_bilateral"     : flagged channels overlap strongly with
                                  {Fp1,Fp2,AFF5h,AFF6h,F1,F2,FC1,FC2}
                                  (general movement / frontalis tension)
      - "chronic_multichannel"  : a very large fraction of ALL channels is
                                  flagged by Z-score (chronic issue, e.g. VP004)
      - "catastrophic_transient": very few channels flagged but extremely
                                  high kurtosis (short, severe events, e.g. VP015)
      - "clean"                 : nothing flagged
      - "other"                 : flagged but doesn't match any known pattern

    Parameters
    ----------
    kurt_bad_ch_names   : list[str] — channel NAMES flagged by kurtosis
    zscore_bad_ch_names : list[str] — channel NAMES flagged by Z-score
    kurt_values         : list[float] — raw kurtosis value per channel
    n_ch                : int — total channel count

    Returns
    -------
    (pattern_label, pattern_score) — pattern_score meaning depends on label:
      temporal_emg / frontal_bilateral -> overlap fraction with the pattern set
      chronic_multichannel             -> fraction of all channels flagged
      catastrophic_transient           -> max |kurtosis| value
      clean / other                    -> 0.0
    """
    bad_set = set(kurt_bad_ch_names)

    if not bad_set and not zscore_bad_ch_names:
        return "clean", 0.0

    # chronic multichannel: Z-score flags a large fraction of ALL channels
    if n_ch > 0 and len(zscore_bad_ch_names) / n_ch > 0.5:
        return "chronic_multichannel", round(len(zscore_bad_ch_names) / n_ch, 3)

    # catastrophic transient: few channels, but extreme kurtosis
    max_k = max((abs(v) for v in kurt_values), default=0.0)
    if max_k > 200 and len(bad_set) <= 6:
        return "catastrophic_transient", round(float(max_k), 1)

    overlap_temporal = (len(bad_set & PATTERN_TEMPORAL_EMG) /
                        len(PATTERN_TEMPORAL_EMG)) if bad_set else 0.0
    overlap_frontal  = (len(bad_set & PATTERN_FRONTAL_BILATERAL) /
                        len(PATTERN_FRONTAL_BILATERAL)) if bad_set else 0.0

    if overlap_temporal >= 0.5 and overlap_temporal >= overlap_frontal:
        return "temporal_emg", round(overlap_temporal, 3)
    if overlap_frontal >= 0.5:
        return "frontal_bilateral", round(overlap_frontal, 3)

    if bad_set:
        return "other", 0.0

    return "clean", 0.0


# ══════════════════════════════════════════════════════════════════════════
#  7. Per-subject visualization
# ══════════════════════════════════════════════════════════════════════════

def visualize_subject(data: np.ndarray, sfreq: int,
                      subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, artifact_type: str, output_dir: str):
    """
    Comprehensive per-subject, per-session figure.
    5 panels: Raw EEG, PSD, Kurtosis, Isolation Forest, Ensemble Vote.
    """
    ch_names = get_ch_names(data.shape[0])
    t = np.arange(data.shape[1]) / sfreq

    fig = plt.figure(figsize=(18, 14))
    fig.suptitle(
        f"EEG Artifact Report — {subject_id} | Session {session}\n"
        f"Quality: {quality['quality_label']} (score={quality['total_score']}/100) "
        f"| pattern: {artifact_type}",
        fontsize=13, fontweight="bold", y=0.98
    )
    gs = gridspec.GridSpec(3, 2, figure=fig,
                           hspace=0.45, wspace=0.35)

    # ── 1. Raw EEG ────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    n_show = min(data.shape[0], 8)   # show up to 8 channels for clarity
    offsets = np.arange(n_show) * float(np.ptp(data[:n_show]) * 0.6 + 1)
    bad_chs = set(stat_r["zscore"]["contaminated_ch"] +
                  stat_r["kurtosis"]["contaminated_ch"])
    for i in range(n_show):
        color = "tomato" if i in bad_chs else "steelblue"
        ax1.plot(t, data[i] + offsets[i], lw=0.4, color=color, alpha=0.8)
    ax1.set_yticks(offsets)
    ax1.set_yticklabels(ch_names[:n_show], fontsize=7)
    ax1.set_xlabel("Time (s)")
    ax1.set_title(f"Raw EEG — first {n_show} channels "
                  f"(red = bad channel)", fontsize=10)
    for at in ml_r["ensemble"]["anomaly_times_s"]:
        ax1.axvspan(at, at + 2.0, color="red", alpha=0.08)
    ax1.axvspan(0, 0, color="red", alpha=0.2, label="ML Anomaly epoch")
    ax1.legend(loc="upper right", fontsize=7)

    # ── 2. PSD, all channels ─────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    for i, sig in enumerate(data):
        f, pxx = welch(sig, fs=sfreq, nperseg=min(512, len(sig)))
        lw = 1.2 if i in bad_chs else 0.5
        ax2.semilogy(f, pxx, lw=lw,
                     color="tomato" if i in bad_chs else "steelblue",
                     alpha=0.6)
    ax2.axvline(50, color="darkred", ls="--", lw=1.2, label="50 Hz notch")
    ax2.set_xlim(0, min(95, sfreq / 2))
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Power (uV^2/Hz)")
    ax2.set_title("PSD — all channels")
    ax2.legend(fontsize=8)

    # ── 3. Kurtosis per channel ───────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    kurt_vals = np.abs(stat_r["kurtosis"]["values"])
    colors_k  = ["tomato" if k > THRESH_KURTOSIS else "steelblue"
                 for k in kurt_vals]
    ax3.bar(range(len(kurt_vals)), kurt_vals, color=colors_k, edgecolor="none")
    ax3.axhline(THRESH_KURTOSIS, color="red", ls="--",
                label=f"thresh={THRESH_KURTOSIS}")
    ax3.set_xlabel("Channel index")
    ax3.set_ylabel("|Kurtosis|")
    ax3.set_title("Kurtosis per Channel")
    ax3.legend(fontsize=8)

    # ── 4. Isolation Forest scores ───────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    times_ml = np.array(ml_r["_times"])
    scores   = np.array(ml_r["isolation_forest"]["scores"])
    thresh_if = np.percentile(scores, 10)
    ax4.plot(times_ml, scores, "o-", ms=2, lw=0.8, color="darkorange")
    ax4.axhline(thresh_if, color="red", ls="--", label="Anomaly threshold")
    ax4.fill_between(times_ml, scores, thresh_if,
                     where=(scores < thresh_if),
                     color="red", alpha=0.25)
    ax4.set_xlabel("Epoch start (s)")
    ax4.set_ylabel("Anomaly Score")
    ax4.set_title("Isolation Forest")
    ax4.legend(fontsize=8)

    # ── 5. Ensemble Vote ─────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    votes = np.array(ml_r["ensemble"]["vote_counts"])
    ax5.bar(times_ml, votes, width=max(0.5, (t[-1] / len(times_ml)) * 0.8),
            color=np.where(votes >= 2, "tomato", "steelblue"))
    ax5.axhline(2, color="red", ls="--", label="Majority (>=2/3)")
    ax5.set_xlabel("Epoch start (s)")
    ax5.set_ylabel("Vote count")
    ax5.set_title("Ensemble Vote (IF + LOF + EE)")
    ax5.legend(fontsize=8)

    path = os.path.join(output_dir,
                        f"{subject_id}_S{session}_artifact_report.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


# ══════════════════════════════════════════════════════════════════════════
#  8. Per-subject text report
# ══════════════════════════════════════════════════════════════════════════

def write_text_report(subject_id: str, session: int,
                      stat_r: dict, sp_r: dict, ml_r: dict,
                      quality: dict, artifact_type: str, artifact_score: float,
                      n_ch: int, output_dir: str):
    ch_names = get_ch_names(n_ch)
    lines    = [
        f"{'='*60}",
        f"EEG Artifact Report — {subject_id} | Session {session}",
        f"{'='*60}",
        f"Overall quality: {quality['quality_label']}  (score = {quality['total_score']}/100)",
        f"Reasons: {' | '.join(quality['reasons']) if quality['reasons'] else 'no issues'}",
        f"Artifact pattern: {artifact_type}  (score={artifact_score})",
        "",
        "-- Statistical ------------------------------------------------",
        f"High Z-score : {[ch_names[i] for i in stat_r['zscore']['contaminated_ch']]} (max={stat_r['zscore']['max_z']:.2f})",
        f"Modified Z   : {[ch_names[i] for i in stat_r['modified_zscore']['contaminated_ch']]}",
        f"High Kurtosis: {[ch_names[i] for i in stat_r['kurtosis']['contaminated_ch']]} (max={stat_r['kurtosis']['max_abs_kurt']:.2f})",
        f"High PTP (>{THRESH_AMP_UV}uV): {[ch_names[i] for i in stat_r['amplitude']['contaminated_ch']]} (max={stat_r['amplitude']['max_ptp_uv']:.1f}uV)",
        f"Variance Jump: {len(stat_r['variance_jump']['contaminated_windows'])} suspicious windows",
        "",
        "-- Signal processing -------------------------------------------",
        f"50Hz noise : {len(sp_r['powerline'])} contaminated channels",
        f"EOG/blinks : {sum(e['blink_count'] for e in sp_r['eog'])} events",
        f"EMG        : {len(sp_r['emg'])} contaminated channels (thresh ratio={THRESH_EMG_RATIO})",
        f"ECG        : {len(sp_r['ecg'])} channels with cardiac-like pattern",
        f"Pop/Jump   : {len(sp_r['electrode_pop'])} channels with jumps",
        f"Flat       : {sp_r['flat']} flat channels",
        f"Wavelet    : {len(sp_r['wavelet'])} suspicious channels",
        f"Residual ICA: {sp_r['ica_residual'].get('n_artifact_components_remaining', '?')} components",
        "",
        "-- Machine learning ----------------------------------------------",
        f"Isolation Forest: {ml_r['isolation_forest']['n_anomalies']} anomaly epochs",
        f"LOF             : {ml_r['lof']['n_anomalies']} anomaly epochs",
        f"Ensemble (>=2/3): {ml_r['ensemble']['n_anomalies_majority']} of {len(ml_r['_times'])} epochs",
        "="*60,
    ]
    path = os.path.join(output_dir,
                        f"{subject_id}_S{session}_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


# ══════════════════════════════════════════════════════════════════════════
#  9. Main pipeline — one subject, one session
# ══════════════════════════════════════════════════════════════════════════

def run_subject_session(subject_id: str, session: int,
                        output_dir: str) -> dict | None:
    """
    Full pipeline for one subject / one session.
    Returns a summary dict, or None if the data could not be found.
    """
    data = load_subject_eeg(subject_id, session)
    if data is None:
        print(f"  [SKIP] {subject_id} S{session} — file not found")
        return None

    n_ch = data.shape[0]
    print(f"  {subject_id} S{session}: {n_ch}ch x {data.shape[1]} samples "
          f"({data.shape[1]/SFREQ:.1f}s) ...", end=" ", flush=True)

    # ── run all detectors ────────────────────────────────────────────────
    stat_r = StatisticalDetector().detect(data, SFREQ)
    sp_det = SignalProcessingDetector(SFREQ)
    sp_r   = sp_det.run_all(data)
    ml_r   = MLDetector().fit_predict(data, SFREQ)
    quality = compute_subject_quality_score(stat_r, sp_r, ml_r, n_ch)

    # ── artifact-pattern classification ────────────────────────────────
    ch_names = get_ch_names(n_ch)
    kurt_bad_names = [ch_names[i] for i in stat_r["kurtosis"]["contaminated_ch"]]
    zscore_bad_names = [ch_names[i] for i in stat_r["zscore"]["contaminated_ch"]]
    artifact_type, artifact_score = classify_artifact_pattern(
        kurt_bad_names, zscore_bad_names,
        stat_r["kurtosis"]["values"], n_ch
    )

    # ── save report and figure ───────────────────────────────────────────
    png_path = visualize_subject(data, SFREQ,
                                 subject_id, session,
                                 stat_r, sp_r, ml_r, quality,
                                 artifact_type, output_dir)
    txt_path = write_text_report(subject_id, session,
                                 stat_r, sp_r, ml_r, quality,
                                 artifact_type, artifact_score,
                                 n_ch, output_dir)

    print(f"[{quality['quality_label']}] score={quality['total_score']} "
          f"pattern={artifact_type}")

    del data
    gc.collect()

    return {
        "subject": subject_id,
        "session": session,
        "quality_score": quality["total_score"],
        "quality_label": quality["quality_label"],
        "reasons": quality["reasons"],
        "artifact_type": artifact_type,
        "artifact_type_score": artifact_score,
        "n_bad_ch_zscore": len(stat_r["zscore"]["contaminated_ch"]),
        "n_bad_ch_kurtosis": len(stat_r["kurtosis"]["contaminated_ch"]),
        "max_ptp_uv": stat_r["amplitude"]["max_ptp_uv"],
        "n_powerline_ch": len(sp_r["powerline"]),
        "n_emg_ch": len(sp_r["emg"]),
        "n_blinks": sum(e["blink_count"] for e in sp_r["eog"]),
        "n_anomaly_epochs_ensemble": ml_r["ensemble"]["n_anomalies_majority"],
        "ica_residual_components": sp_r["ica_residual"].get("n_artifact_components_remaining", -1),
        "png": png_path,
        "txt": txt_path,
    }


# ══════════════════════════════════════════════════════════════════════════
#  10. Pipeline for all 26 subjects
# ══════════════════════════════════════════════════════════════════════════

def run_all_subjects(subjects: list[str] = ALL_SUBJECTS,
                     sessions: list[int] | None = None,
                     output_dir: str = OUTPUT_DIR) -> list[dict]:
    """
    Run the full pipeline over all subjects and sessions.

    Parameters
    ----------
    subjects   : list of subject IDs (default: all 26)
    sessions   : list of session numbers (default: [1, 2, 3])
    output_dir : output folder
    """
    if sessions is None:
        sessions = list(range(1, N_SESSIONS + 1))

    os.makedirs(output_dir, exist_ok=True)

    print("=" * 60)
    print(f"  EEG Artifact Detection — {len(subjects)} subjects x {len(sessions)} sessions")
    print("=" * 60)

    all_results = []

    for subj in subjects:
        print(f"\n[{subj}]")
        for sess in sessions:
            result = run_subject_session(subj, sess, output_dir)
            if result:
                all_results.append(result)

    # ── summary CSV / JSON / plots across all subjects ─────────────────
    if all_results:
        _save_summary_csv(all_results, output_dir)
        _save_summary_json(all_results, output_dir)
        _plot_summary_heatmap(all_results, output_dir)
        _plot_pattern_summary(all_results, output_dir)

    print(f"\n{'='*60}")
    print(f"  Done — {len(all_results)} sessions processed")
    print(f"  Output: {output_dir}")
    print("=" * 60)

    return all_results


def _save_summary_csv(results: list[dict], output_dir: str):
    """Save summary CSV"""
    path = os.path.join(output_dir, "summary_all_subjects.csv")
    if not results:
        return
    fieldnames = [k for k in results[0].keys() if k not in ("png", "txt")]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow({k: r[k] for k in fieldnames})
    print(f"\n  [CSV] {path}")


def _save_summary_json(results: list[dict], output_dir: str):
    """Save full summary JSON"""
    path = os.path.join(output_dir, "summary_all_subjects.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"  [JSON] {path}")


def _plot_summary_heatmap(results: list[dict], output_dir: str):
    """
    Quality heatmap: subject x session.
    Color: green = good, red = poor.
    """
    subjects = sorted(set(r["subject"] for r in results))
    sessions = sorted(set(r["session"] for r in results))

    score_mat = np.full((len(subjects), len(sessions)), np.nan)
    for r in results:
        si = subjects.index(r["subject"])
        sj = sessions.index(r["session"])
        score_mat[si, sj] = r["quality_score"]

    fig, ax = plt.subplots(figsize=(max(6, len(sessions) * 1.5),
                                    max(8, len(subjects) * 0.45)))
    im = ax.imshow(score_mat, cmap="RdYlGn_r", vmin=0, vmax=80, aspect="auto")
    plt.colorbar(im, ax=ax, label="Artifact Score (higher = worse)")

    ax.set_xticks(range(len(sessions)))
    ax.set_xticklabels([f"S{s}" for s in sessions])
    ax.set_yticks(range(len(subjects)))
    ax.set_yticklabels(subjects, fontsize=7)

    for i in range(len(subjects)):
        for j in range(len(sessions)):
            if not np.isnan(score_mat[i, j]):
                ax.text(j, i, f"{score_mat[i,j]:.0f}",
                        ha="center", va="center", fontsize=7,
                        color="black")

    ax.set_title("EEG Quality Heatmap — all subjects\n"
                 "(green = good | red = poor)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, "summary_heatmap.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [Heatmap] {path}")


def _plot_pattern_summary(results: list[dict], output_dir: str):
    """
    NEW: bar chart summarizing how many sessions fall into each
    artifact-pattern category, plus a table listing which subjects
    belong to each category.
    """
    pattern_counts: dict[str, int] = {}
    pattern_subjects: dict[str, set] = {}
    for r in results:
        p = r.get("artifact_type", "unknown")
        pattern_counts[p] = pattern_counts.get(p, 0) + 1
        pattern_subjects.setdefault(p, set()).add(r["subject"])

    order = sorted(pattern_counts, key=lambda k: -pattern_counts[k])
    counts = [pattern_counts[p] for p in order]

    color_map = {
        "clean": "#2ca02c", "temporal_emg": "#d62728",
        "frontal_bilateral": "#ff7f0e", "chronic_multichannel": "#8c1616",
        "catastrophic_transient": "#9467bd", "other": "#7f7f7f",
    }
    colors = [color_map.get(p, "#1f77b4") for p in order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(order, counts, color=colors, edgecolor="white")
    for bar, p in zip(bars, order):
        n_subj = len(pattern_subjects[p])
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{bar.get_height():.0f}\n({n_subj} subj.)",
                ha="center", fontsize=8)
    ax.set_ylabel("Number of sessions")
    ax.set_title("Artifact Pattern Distribution — all sessions")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    path = os.path.join(output_dir, "artifact_pattern_summary.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  [Pattern summary] {path}")

    # text table too
    lines = ["Artifact pattern distribution", "=" * 40]
    for p in order:
        lines.append(f"{p:24s}: {pattern_counts[p]:3d} sessions | "
                     f"subjects: {sorted(pattern_subjects[p])}")
    txt_path = os.path.join(output_dir, "artifact_pattern_summary.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"  [Pattern table] {txt_path}")


# ══════════════════════════════════════════════════════════════════════════
#  Entry point
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Option A: all subjects, all sessions ─────────────────────────────
    results = run_all_subjects()

    # ── Option B: a single subject (for testing) ─────────────────────────
    # results = run_all_subjects(subjects=["VP001"], sessions=[1])

    # ── Option C: specific sessions only ──────────────────────────────────
    # results = run_all_subjects(sessions=[1])

    # ── quick terminal summary ───────────────────────────────────────────
    print("\nQuality summary:")
    print(f"{'Subject':8s} {'Session':8s} {'Score':8s} {'Label':10s} {'Pattern':22s} {'Reasons'}")
    print("-" * 100)
    for r in sorted(results, key=lambda x: x["quality_score"], reverse=True):
        print(f"{r['subject']:8s} {r['session']:<8d} {r['quality_score']:<8.1f} "
              f"{r['quality_label']:10s} {r['artifact_type']:22s} "
              f"{'; '.join(r['reasons'][:2])}")


  EEG Artifact Detection — 26 subjects x 3 sessions

[VP001]
  VP001 S1: 28ch x 131644 samples (658.2s) ... [moderate] score=38.8 pattern=temporal_emg
  VP001 S2: 28ch x 131644 samples (658.2s) ... [suspicious] score=50.8 pattern=temporal_emg
  VP001 S3: 28ch x 131644 samples (658.2s) ... [good] score=1.9 pattern=clean

[VP002]
  VP002 S1: 28ch x 131747 samples (658.7s) ... [good] score=1.9 pattern=clean
  VP002 S2: 28ch x 131746 samples (658.7s) ... [good] score=1.9 pattern=clean
  VP002 S3: 28ch x 131747 samples (658.7s) ... [good] score=1.7 pattern=clean

[VP003]
  VP003 S1: 28ch x 129748 samples (648.7s) ... [good] score=2.0 pattern=clean
  VP003 S2: 28ch x 129747 samples (648.7s) ... [moderate] score=26.8 pattern=other
  VP003 S3: 28ch x 129748 samples (648.7s) ... [good] score=1.6 pattern=other

[VP004]
  VP004 S1: 28ch x 129657 samples (648.3s) ... [poor] score=80.5 pattern=chronic_multichannel
  VP004 S2: 28ch x 129657 samples (648.3s) ... [poor] score=81.9 pattern=chronic_mult